# Continuous BMI: immune-coarse reference model

The v2.2 model uses cell-type-specific Student-t tail parameters. This all-profile fit supplies the reference for the low-read-excluded primary analysis.

Set BASE, PCA_DIR_OVERRIDE and MODEL_ROOT to the input and new output directories. Set NOTEBOOK_PATH_OVERRIDE to the saved path of this curated notebook. Select RUN_SPEC, PROFILE and WORKFLOW_STAGE. Execute installation, mounting and configuration, then the cells for the selected stage. Run the paired PPC audit only after the trace and PPC arrays exist.


In [ ]:
# 1. Install the validated sampling environment
# Run first in a fresh CPU High-RAM Colab runtime, before importing PyMC.
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "pymc==5.28.5",
    "arviz==0.23.4",
    "nutpie==0.16.11",
])
print("Pinned Bayesian environment installed.")


In [ ]:
# 2. Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
# 3. Configuration, run registry, gates, and publication style
from __future__ import annotations

import gc
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import re
import shutil
import warnings
from datetime import datetime, timezone
from pathlib import Path

import arviz as az
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
import xarray as xr
from IPython.display import Markdown, display
from PIL import Image
from scipy.stats import beta as beta_dist, norm

warnings.filterwarnings("default")
pd.set_option("display.max_columns", 160)

TESTED_PYMC_VERSION = "5.28.5"
TESTED_ARVIZ_VERSION = "0.23.4"
TESTED_NUTPIE_VERSION = "0.16.11"
if pm.__version__ != TESTED_PYMC_VERSION:
    warnings.warn(
        f"Validated against PyMC {TESTED_PYMC_VERSION}; runtime has {pm.__version__}."
    )
if az.__version__ != TESTED_ARVIZ_VERSION:
    warnings.warn(
        f"Validated against ArviZ {TESTED_ARVIZ_VERSION}; runtime has {az.__version__}."
    )
runtime_nutpie_version = importlib.metadata.version("nutpie")
if runtime_nutpie_version != TESTED_NUTPIE_VERSION:
    warnings.warn(
        f"Validated against nutpie {TESTED_NUTPIE_VERSION}; runtime has "
        f"{runtime_nutpie_version}."
    )

ANALYSIS_VERSION = (
    "normalized-state-bhm-v2.2-separate-compartments-"
    "global-patient-fixed-loading-celltype-nu"
)
MODEL_ARCHITECTURE = "v2p2_g1_fixed_ctnu"
GATE_VERSION = "three-stage-v1"
PPC_VERSION = "residual-symmetric-v2"
REVEAL_VERSION = "rope-bfdr-mc-precision-v1"
NOTEBOOK_RELEASE_SHA256 = "f4f4a3b3a6d7bdb0ef535214866eeb14b2cfd5b1dded03083e69c817e611ce32"

BASE = Path("/content/drive/MyDrive/new_dysfunction")
AUDIT_ROOT = BASE / "bayesprism_initial_issue_verification_v1"
PCA_ROOT = AUDIT_ROOT / "publication_pca_tiered_qc_v1"
MODEL_ROOT = BASE / "normalized_state_bhm_v2p2_celltype_nu"

PCA_DIR_OVERRIDE = "/content/drive/MyDrive/new_dysfunction/bayesprism_initial_issue_verification_v1/publication_pca_tiered_qc_v1/20260906T013040524557Z"
NOTEBOOK_PATH_OVERRIDE = None
RUN_SPEC = "immune_coarse_primary"
PROFILE = "publication"  # smoke | standard | publication | publication_long
WORKFLOW_STAGE = "ppc_review"  # preflight | fit | diagnostics | ppc | ppc_review | reveal | visualize | aggregate
REVEAL_RESULTS_APPROVED = False
VISUALIZATION_APPROVED = False
RUN_TRIAGE = False
PPC_JOINT_DRAWS_OVERRIDE = None  # only to resolve a near-threshold formal PPC

VALID_WORKFLOW_STAGES = {
    "preflight", "fit", "diagnostics", "ppc", "ppc_review", "reveal",
    "visualize", "aggregate",
}
if WORKFLOW_STAGE not in VALID_WORKFLOW_STAGES:
    raise ValueError(f"WORKFLOW_STAGE must be one of {sorted(VALID_WORKFLOW_STAGES)}")

RUN_MODEL = WORKFLOW_STAGE == "fit"
RUN_PRIOR_CHECK = WORKFLOW_STAGE == "preflight"
RUN_PPC = WORKFLOW_STAGE == "ppc"
RUN_PPC_REVIEW = WORKFLOW_STAGE == "ppc_review"
RUN_DIAGNOSTIC_FIGURES = WORKFLOW_STAGE == "diagnostics"
RUN_RESULT_REVEAL = WORKFLOW_STAGE == "reveal"
RUN_BIOLOGICAL_FIGURES = WORKFLOW_STAGE == "visualize"
RUN_PRIMARY_AGGREGATION = WORKFLOW_STAGE == "aggregate"
LOAD_TRACE_STAGES = {"diagnostics", "ppc", "ppc_review", "reveal", "visualize"}

LIKELIHOOD = "student_t"
NUTS_BACKEND = "nutpie"
BASE_SEED = 20260904

PRIMARY_SCORE_VARIANT = "within_component_logcpm_z_cap5"
CAP3_SCORE_VARIANT = "within_component_logcpm_z_cap3"
CLUSTER_REFERENCE_VARIANT = PRIMARY_SCORE_VARIANT
IMMUNE_COARSE_TIERED = (("immune_course", "tiered_primary_keep"),)
IMMUNE_COARSE_STRICT = (("immune_course", "strict_qc_keep"),)
IMMUNE_FINE_TIERED = (("immune_fine", "tiered_primary_keep"),)
IMMUNE_FINE_STRICT = (("immune_fine", "strict_qc_keep"),)
NONIMMUNE_STRICT = (("non_immune", "strict_qc_keep"),)
NONIMMUNE_TIERED = (("non_immune", "tiered_primary_keep"),)

RUN_SPECS = {
    "immune_coarse_primary": {
        "role": "primary", "analysis_family": "immune_coarse",
        "components": IMMUNE_COARSE_TIERED,
        "expected_celltypes": 8, "expected_features": 306,
        "score_variant": PRIMARY_SCORE_VARIANT, "feature_mode": "all",
        "bmi_shape": "linear", "include_theta": False,
    },
    "immune_fine_secondary": {
        "role": "secondary", "analysis_family": "immune_fine",
        "components": IMMUNE_FINE_TIERED,
        "expected_celltypes": 14, "expected_features": 496,
        "score_variant": PRIMARY_SCORE_VARIANT, "feature_mode": "all",
        "bmi_shape": "linear", "include_theta": False,
    },
    "nonimmune_primary": {
        "role": "primary", "analysis_family": "nonimmune",
        "components": NONIMMUNE_STRICT,
        "expected_celltypes": 4, "expected_features": 112,
        "score_variant": PRIMARY_SCORE_VARIANT, "feature_mode": "all",
        "bmi_shape": "linear", "include_theta": False,
    },
    "immune_coarse_strict_qc_sensitivity": {
        "role": "sensitivity", "analysis_family": "immune_coarse",
        "components": IMMUNE_COARSE_STRICT,
        "expected_celltypes": 6, "expected_features": 232,
        "score_variant": PRIMARY_SCORE_VARIANT, "feature_mode": "all",
        "bmi_shape": "linear", "include_theta": False,
    },
    "immune_fine_strict_qc_sensitivity": {
        "role": "sensitivity", "analysis_family": "immune_fine",
        "components": IMMUNE_FINE_STRICT,
        "expected_celltypes": 12, "expected_features": 430,
        "score_variant": PRIMARY_SCORE_VARIANT, "feature_mode": "all",
        "bmi_shape": "linear", "include_theta": False,
    },
    "nonimmune_tiered_qc_sensitivity": {
        "role": "sensitivity", "analysis_family": "nonimmune",
        "components": NONIMMUNE_TIERED,
        "expected_celltypes": 6, "expected_features": 157,
        "score_variant": PRIMARY_SCORE_VARIANT, "feature_mode": "all",
        "bmi_shape": "linear", "include_theta": False,
    },
}

for family, components, n_celltypes, n_features in (
    ("immune_coarse", IMMUNE_COARSE_TIERED, 8, 306),
    ("immune_fine", IMMUNE_FINE_TIERED, 14, 496),
    ("nonimmune", NONIMMUNE_STRICT, 4, 112),
):
    key_prefix = "nonimmune" if family == "nonimmune" else family
    RUN_SPECS[f"{key_prefix}_cap3_sensitivity"] = {
        "role": "sensitivity", "analysis_family": family,
        "components": components, "expected_celltypes": n_celltypes,
        "expected_features": n_features, "score_variant": CAP3_SCORE_VARIANT,
        "feature_mode": "all", "bmi_shape": "linear", "include_theta": False,
    }
    RUN_SPECS[f"{key_prefix}_theta_adjusted_sensitivity"] = {
        "role": "sensitivity", "analysis_family": family,
        "components": components, "expected_celltypes": n_celltypes,
        "expected_features": n_features, "score_variant": PRIMARY_SCORE_VARIANT,
        "feature_mode": "all", "bmi_shape": "linear", "include_theta": True,
    }
    RUN_SPECS[f"{key_prefix}_quadratic_sensitivity"] = {
        "role": "sensitivity", "analysis_family": family,
        "components": components, "expected_celltypes": n_celltypes,
        "expected_features": n_features, "score_variant": PRIMARY_SCORE_VARIANT,
        "feature_mode": "all", "bmi_shape": "linear_quadratic", "include_theta": False,
    }
    RUN_SPECS[f"{key_prefix}_cluster_representative_sensitivity"] = {
        "role": "sensitivity", "analysis_family": family,
        "components": components, "expected_celltypes": n_celltypes,
        "expected_features": n_features, "score_variant": PRIMARY_SCORE_VARIANT,
        "feature_mode": "cluster_representative", "bmi_shape": "linear",
        "include_theta": False,
    }

PROFILES = {
    "smoke": {"tune": 500, "draws": 500, "chains": 2, "target_accept": 0.97},
    "standard": {"tune": 1000, "draws": 1000, "chains": 4, "target_accept": 0.98},
    "publication": {"tune": 2000, "draws": 4000, "chains": 4, "target_accept": 0.99},
    "publication_long": {"tune": 2000, "draws": 8000, "chains": 4, "target_accept": 0.99},
}
PROFILE_PREDECESSOR = {
    "smoke": None, "standard": "smoke", "publication": "standard",
    "publication_long": "publication",
}

if RUN_SPEC not in RUN_SPECS:
    raise ValueError(f"Unknown RUN_SPEC {RUN_SPEC!r}; choose from {sorted(RUN_SPECS)}")
if PROFILE not in PROFILES:
    raise ValueError(f"Unknown PROFILE {PROFILE!r}")
ACTIVE_SPEC = RUN_SPECS[RUN_SPEC]
if ACTIVE_SPEC["analysis_family"] == "nonimmune":
    warnings.warn(
        "Non-immune already passed corrected PPC with the v2.1c global-nu model. "
        "Use this v2.2 non-immune run only if you explicitly want an optional "
        "cell-type-nu sensitivity."
    )
SCORE_VARIANT = ACTIVE_SPEC["score_variant"]
FEATURE_MODE = ACTIVE_SPEC["feature_mode"]
BMI_SHAPE = ACTIVE_SPEC["bmi_shape"]
INCLUDE_THETA = ACTIVE_SPEC["include_theta"]
if SCORE_VARIANT not in {PRIMARY_SCORE_VARIANT, CAP3_SCORE_VARIANT}:
    raise ValueError("Only audited within-component endpoints are allowed")
if FEATURE_MODE not in {"all", "cluster_representative"}:
    raise ValueError("FEATURE_MODE must be all or cluster_representative")
if BMI_SHAPE not in {"linear", "linear_quadratic"}:
    raise ValueError("BMI_SHAPE must be linear or linear_quadratic")
if LIKELIHOOD != "student_t":
    raise ValueError("The locked model uses a Student-t likelihood")

# Priors on the feature-standardized endpoint.
BASELINE_SD = 0.50
CELLTYPE_BMI_SD = 0.20
FEATURE_BMI_SD = 0.30
CELLTYPE_COVARIATE_SD = 0.25
FEATURE_COVARIATE_SD = 0.25
CELLTYPE_QUADRATIC_SD = 0.15
FEATURE_QUADRATIC_SD = 0.20
CELLTYPE_THETA_SD = 0.25
FEATURE_THETA_SD = 0.25
PATIENT_GLOBAL_SD = 0.50
LOG_SIGMA_LOCATION = math.log(0.65)
LOG_SIGMA_LOCATION_SD = 0.40
CELLTYPE_LOG_SIGMA_SD = 0.25
FEATURE_LOG_SIGMA_SD = 0.50
NU_MINUS_TWO_RATE = 0.10

ROPE_GRID = (0.05, 0.10, 0.15, 0.20)
PRIMARY_ROPE = 0.10
Q_LEVEL = 0.05
PRIOR_DRAWS = 200

# Three-stage convergence gates.
MAX_TREE_DEPTH = 10
MAX_TREE_DEPTH_FRACTION = 0.01
BFMI_MIN = 0.30
SMOKE_GLOBAL_RHAT_HARD_MAX = 1.10
SMOKE_GLOBAL_ESS_HARD_MIN = 20
SMOKE_SMALL_RHAT_MAX = 1.03
SMOKE_SMALL_ESS_MIN = 100
SMOKE_ESTIMAND_RHAT_MAX = 1.03
SMOKE_ESTIMAND_ESS_MIN = 100
SMOKE_ESTIMAND_BAD_FRACTION_MAX = 0.005
SMOKE_ESTIMAND_RHAT_HARD_MAX = 1.05
SMOKE_ESTIMAND_ESS_HARD_MIN = 50
SMOKE_VECTOR_RHAT_MAX = 1.05
SMOKE_VECTOR_ESS_MIN = 100
SMOKE_VECTOR_BAD_FRACTION_MAX = 0.01
FINAL_RHAT_MAX = 1.01
FINAL_ESS_MIN = 400

# Joint posterior-predictive calibration/evaluation design.
PPC_SETTINGS = {
    "smoke": {"joint_draws": 200, "calibration_draws": 0, "evaluation_draws": 0, "formal_decision": False},
    "standard": {"joint_draws": 2000, "calibration_draws": 1000, "evaluation_draws": 1000, "formal_decision": True},
    "publication": {"joint_draws": 4000, "calibration_draws": 2000, "evaluation_draws": 2000, "formal_decision": True},
    "publication_long": {"joint_draws": 8000, "calibration_draws": 4000, "evaluation_draws": 4000, "formal_decision": True},
}
PPC_BATCH_MEMORY_MB = 64
PPC_REFERENCE_ALPHA = 0.05
PPC_OMNIBUS_ALPHA = 0.025
PPC_CORRELATION_THRESHOLD = 0.90

# Publication reveal precision.
ROPE_INDICATOR_ESS_MIN = 400.0
ROPE_LOCAL_NULL_MCSE_MAX = 0.005
BFDR_MCSE_MAX = 0.0025
MC_ALPHA = 0.025

def stable_run_seed(spec_name: str) -> int:
    offset = int(hashlib.sha256(spec_name.encode()).hexdigest()[:8], 16) % 1_000_000
    return BASE_SEED + offset

RUN_SEED = stable_run_seed(RUN_SPEC)

def make_run_suffix(spec_name: str = RUN_SPEC, profile: str = PROFILE) -> str:
    spec = RUN_SPECS[spec_name]
    return (
        f"{MODEL_ARCHITECTURE}__{spec_name}__{spec['score_variant']}__"
        f"{spec['feature_mode']}__{spec['bmi_shape']}__"
        f"theta_{int(spec['include_theta'])}__{profile}"
    )

def run_directory_for(spec_name: str = RUN_SPEC, profile: str = PROFILE) -> Path:
    return MODEL_ROOT / make_run_suffix(spec_name, profile)

OUTPUT_DIR = run_directory_for()
try:
    component_limit = os.pathconf(str(MODEL_ROOT.parent), "PC_NAME_MAX")
except (OSError, ValueError):
    component_limit = 255
if len(OUTPUT_DIR.name.encode()) > component_limit:
    raise OSError(f"Run-directory basename exceeds PC_NAME_MAX={component_limit}: {OUTPUT_DIR.name}")
if len(str(OUTPUT_DIR)) >= 1000:
    raise OSError("Run path is unexpectedly long; move MODEL_ROOT to a shorter path")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIGURE_STYLE = "legacy_exact"
FIGURE_DPI = 600
FIGURE_COLORS = {
    "positive": "#D62728", "negative": "#2166AC",
    "immune_coarse": "#32CD32", "immune_fine": "#4169E1",
    "nonimmune": "#8B4513", "neutral": "#666666",
}
HEATMAP_CMAP = "RdBu_r"
FIGURE_ROOT = OUTPUT_DIR / "figures"
FIGURE_DIRECTORIES = {
    category: {"tiff": FIGURE_ROOT / category / "tiff", "svg": FIGURE_ROOT / category / "svg"}
    for category in ("diagnostics", "results")
}
for formats in FIGURE_DIRECTORIES.values():
    for directory in formats.values():
        directory.mkdir(parents=True, exist_ok=True)

def apply_publication_style() -> None:
    mpl.rcParams.update({
        "font.family": "DejaVu Sans", "figure.facecolor": "white",
        "axes.facecolor": "white", "axes.edgecolor": "black",
        "axes.linewidth": 1.5, "axes.titleweight": "bold", "axes.titlesize": 18,
        "axes.labelsize": 16, "axes.labelweight": "bold", "xtick.labelsize": 14,
        "ytick.labelsize": 14, "legend.fontsize": 11, "grid.color": "#B8B8B8",
        "grid.linestyle": "--", "grid.linewidth": 0.7, "grid.alpha": 0.35,
        "savefig.facecolor": "white", "savefig.edgecolor": "none", "svg.fonttype": "none",
    })

def save_publication_figure(fig, stem: str, category: str) -> None:
    if category not in FIGURE_DIRECTORIES:
        raise ValueError(f"Unknown figure category: {category}")
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", stem).strip("_")
    tiff_path = FIGURE_DIRECTORIES[category]["tiff"] / f"{safe}.tiff"
    svg_path = FIGURE_DIRECTORIES[category]["svg"] / f"{safe}.svg"
    temporary = tiff_path.with_name(f".{safe}.rendering.tiff")
    finalizing = tiff_path.with_name(f".{safe}.finalizing.tiff")
    for path in (temporary, finalizing):
        if path.exists(): path.unlink()
    fig.savefig(svg_path, format="svg", bbox_inches="tight")
    compression = "tiff_lzw"
    try:
        fig.savefig(temporary, dpi=FIGURE_DPI, format="tiff", bbox_inches="tight", pil_kwargs={"compression": compression})
        with Image.open(temporary) as image: image.verify()
    except Exception as error:
        warnings.warn(f"LZW TIFF validation failed ({error}); using uncompressed TIFF")
        if temporary.exists(): temporary.unlink()
        compression = "raw"
        fig.savefig(temporary, dpi=FIGURE_DPI, format="tiff", bbox_inches="tight")
        with Image.open(temporary) as image: image.verify()
    shutil.copyfile(temporary, finalizing)
    with Image.open(finalizing) as image: image.verify()
    finalizing.replace(tiff_path)
    temporary.unlink()
    print("Saved:", tiff_path, f"({compression})")
    print("Saved:", svg_path)

def cleanup_figure_rendering_temps() -> None:
    gc.collect()
    for path in list(FIGURE_ROOT.rglob(".*.rendering.tiff")) + list(FIGURE_ROOT.rglob(".*.finalizing.tiff")):
        try: path.unlink()
        except FileNotFoundError: pass

apply_publication_style()
print("Python", platform.python_version(), "| PyMC", pm.__version__, "| ArviZ", az.__version__)
print("Run:", RUN_SPEC, "| profile:", PROFILE, "| stage:", WORKFLOW_STAGE)
print("Architecture:", MODEL_ARCHITECTURE, "| seed:", RUN_SEED)
print("Output:", OUTPUT_DIR)


In [ ]:
# 4. Resolve audited inputs and write a deterministic preflight manifest
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

def sha256_json(value) -> str:
    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), default=str).encode()
    return hashlib.sha256(payload).hexdigest()

def atomic_csv(frame: pd.DataFrame, path: Path, **kwargs) -> None:
    temporary = path.with_name(path.name + ".tmp")
    frame.to_csv(temporary, index=False, **kwargs)
    temporary.replace(path)

def atomic_text(value: str, path: Path) -> None:
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(value, encoding="utf-8")
    temporary.replace(path)

def atomic_json(value, path: Path) -> None:
    atomic_text(json.dumps(value, indent=2, sort_keys=True, default=str), path)

def strict_bool_series(values: pd.Series, label: str) -> pd.Series:
    true_values = {True, 1, "1", "true", "t", "yes", "y"}
    false_values = {False, 0, "0", "false", "f", "no", "n"}
    parsed = []
    for value in values:
        normalized = value.strip().lower() if isinstance(value, str) else value
        if normalized in true_values:
            parsed.append(True)
        elif normalized in false_values:
            parsed.append(False)
        else:
            raise ValueError(f"{label}: unrecognized Boolean value {value!r}")
    return pd.Series(parsed, index=values.index, dtype=bool)

MUTABLE_NOTEBOOK_ASSIGNMENTS = {
    "PCA_DIR_OVERRIDE", "NOTEBOOK_PATH_OVERRIDE", "RUN_SPEC", "PROFILE",
    "WORKFLOW_STAGE", "REVEAL_RESULTS_APPROVED", "VISUALIZATION_APPROVED",
    "RUN_TRIAGE", "PPC_JOINT_DRAWS_OVERRIDE", "NOTEBOOK_RELEASE_SHA256",
}

def notebook_code_sha256(path: Path) -> str:
    notebook = json.loads(path.read_text(encoding="utf-8"))
    normalized_sources = []
    assignment = re.compile(
        r"^(" + "|".join(sorted(MUTABLE_NOTEBOOK_ASSIGNMENTS)) + r")\s*=.*$"
    )
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        lines = []
        for line in "".join(cell.get("source", [])).splitlines():
            match = assignment.match(line)
            lines.append(f"{match.group(1)} = <MUTABLE>" if match else line.rstrip())
        normalized_sources.append("\n".join(lines).rstrip())
    return hashlib.sha256("\n\n# CELL\n\n".join(normalized_sources).encode()).hexdigest()

def verify_notebook_release() -> dict:
    if NOTEBOOK_PATH_OVERRIDE is None:
        if RUN_MODEL or PROFILE in {"standard", "publication", "publication_long"} or WORKFLOW_STAGE in {"reveal", "visualize"}:
            raise ValueError(
                "Set NOTEBOOK_PATH_OVERRIDE to this saved v2.2 notebook before "
                "any fit, standard/publication check, reveal, or visualization."
            )
        return {"status": "embedded_release_unverified", "sha256": NOTEBOOK_RELEASE_SHA256}
    path = Path(NOTEBOOK_PATH_OVERRIDE)
    if not path.exists():
        raise FileNotFoundError(path)
    actual = notebook_code_sha256(path)
    if actual != NOTEBOOK_RELEASE_SHA256:
        raise ValueError(
            "Notebook code differs from the validated v2.2 release. Save/re-upload "
            "the unmodified notebook; only declared user switches may change."
        )
    return {"status": "verified", "path": str(path), "sha256": actual}

NOTEBOOK_RELEASE = {
    "status": "development_smoke_unverified",
    "sha256": NOTEBOOK_RELEASE_SHA256,
}

def latest_complete_directory(parent: Path, required) -> Path:
    if not parent.exists():
        raise FileNotFoundError(parent)
    candidates = sorted(
        (path for path in parent.iterdir() if path.is_dir()),
        key=lambda path: path.name,
        reverse=True,
    )
    for candidate in candidates:
        if all((candidate / name).exists() for name in required):
            return candidate
    raise FileNotFoundError(f"No complete run found under {parent}")

if RUN_PRIMARY_AGGREGATION:
    print('Aggregation stage: current-run data resolution is intentionally skipped.')
else:
    PCA_REQUIRED_FILES = (
        "pca_manifest.json",
        "component_inclusion_tiers.csv",
        "pca_tiered_vs_strict_stability.csv",
        "pca_input_hashes.csv",
    )
    if PCA_DIR_OVERRIDE is None:
        if PROFILE in {"publication", "publication_long"} or WORKFLOW_STAGE in {"reveal", "visualize"}:
            raise ValueError("Publication/reveal requires an exact PCA_DIR_OVERRIDE")
        PCA_DIR = latest_complete_directory(PCA_ROOT, required=PCA_REQUIRED_FILES)
    else:
        PCA_DIR = Path(PCA_DIR_OVERRIDE)
        missing_pca_files = [
            name for name in PCA_REQUIRED_FILES if not (PCA_DIR / name).exists()
        ]
        if missing_pca_files:
            raise FileNotFoundError(
                f"PCA_DIR_OVERRIDE is incomplete: {missing_pca_files}"
            )
    pca_manifest_path = PCA_DIR / "pca_manifest.json"
    pca_manifest = json.loads(pca_manifest_path.read_text(encoding="utf-8"))
    REVIEW_DIR = Path(pca_manifest["combined_review_directory"])
    combined_manifest_path = REVIEW_DIR / "combined_review_manifest.json"
    combined_manifest = json.loads(combined_manifest_path.read_text(encoding="utf-8"))
    RUN_DIRS = {
        compartment: Path(path)
        for compartment, path in combined_manifest["selected_runs"].items()
    }

    tiers_path = PCA_DIR / "component_inclusion_tiers.csv"
    pca_stability_path = PCA_DIR / "pca_tiered_vs_strict_stability.csv"
    cluster_membership_path = REVIEW_DIR / "normalized_correlation_cluster_membership.csv"
    pca_hashes_path = PCA_DIR / "pca_input_hashes.csv"
    tiers = pd.read_csv(tiers_path)
    pca_stability = pd.read_csv(pca_stability_path)
    cluster_membership = pd.read_csv(cluster_membership_path)
    pca_hashes = pd.read_csv(pca_hashes_path)

    # Verify that the audit score files have not changed since PCA.
    hash_mismatches = []
    for row in pca_hashes.itertuples(index=False):
        source = Path(row.path)
        if not source.exists():
            hash_mismatches.append((str(source), "missing"))
        elif sha256_file(source) != row.sha256:
            hash_mismatches.append((str(source), "sha256_changed"))
    if hash_mismatches:
        raise ValueError(f"PCA input provenance changed: {hash_mismatches}")

    def choose_cluster_representatives(
        compartment: str,
        run_dir: Path,
        candidate_features: set[str],
    ) -> set[str]:
        clusters = cluster_membership[
            cluster_membership["compartment"].eq(compartment)
            & cluster_membership["variant"].eq(CLUSTER_REFERENCE_VARIANT)
            & cluster_membership["correlation_kind"].eq(
                "residual_bmi_age_sex_theta"
            )
            & cluster_membership["feature"].isin(candidate_features)
        ][["feature", "cluster_id"]].drop_duplicates()
        if clusters["feature"].duplicated().any():
            raise ValueError(f"{compartment}: a feature belongs to multiple clusters")

        missing_cluster = sorted(candidate_features - set(clusters["feature"]))
        if missing_cluster:
            clusters = pd.concat(
                [
                    clusters,
                    pd.DataFrame(
                        {
                            "feature": missing_cluster,
                            "cluster_id": [
                                f"{compartment}__singleton__{feature}"
                                for feature in missing_cluster
                            ],
                        }
                    ),
                ],
                ignore_index=True,
            )

        metadata_path = run_dir / "score_metadata_all_variants.csv"
        metadata = pd.read_csv(metadata_path)
        metadata = metadata[
            metadata["variant"].eq(CLUSTER_REFERENCE_VARIANT)
            & strict_bool_series(metadata["included"], "score_metadata.included")
        ][["feature", "n_genes_available", "coverage"]].drop_duplicates()
        if metadata["feature"].duplicated().any():
            raise ValueError(f"{compartment}: ambiguous score metadata")

        ranked = clusters.merge(metadata, on="feature", how="left", validate="one_to_one")
        ranked["n_genes_available"] = pd.to_numeric(
            ranked["n_genes_available"], errors="coerce"
        ).fillna(-1)
        ranked["coverage"] = pd.to_numeric(ranked["coverage"], errors="coerce").fillna(-1)
        ranked = ranked.sort_values(
            ["cluster_id", "n_genes_available", "coverage", "feature"],
            ascending=[True, False, False, True],
        )
        representatives = ranked.drop_duplicates("cluster_id", keep="first")
        return set(representatives["feature"])

    cohort_reference = None
    analysis_parts = []
    component_inventory_parts = []
    source_hash_rows = []
    source_feature_count_before_mode = 0

    for compartment, tier_column in RUN_SPECS[RUN_SPEC]["components"]:
        run_dir = RUN_DIRS[compartment]
        cohort_path = run_dir / "analysis_cohort.csv"
        score_path = run_dir / "audit_scores_all_variants.csv.gz"
        patient_qc_path = run_dir / "patient_qc.csv"

        cohort = pd.read_csv(cohort_path, dtype={"base_sample_id": str})
        scores = pd.read_csv(score_path, dtype={"base_sample_id": str})
        patient_qc = pd.read_csv(patient_qc_path, dtype={"base_sample_id": str})

        required_cohort = {
            "base_sample_id", "bmi", "bmi_z", "bmi_group", "age_z", "sex_centered"
        }
        required_scores = {
            "base_sample_id", "CellType", "Signature", "feature", "variant", "score"
        }
        if not required_cohort.issubset(cohort.columns):
            raise ValueError(f"{compartment}: incomplete cohort table")
        if not required_scores.issubset(scores.columns):
            raise ValueError(f"{compartment}: incomplete audit score table")
        if cohort["base_sample_id"].nunique() != 127 or cohort["base_sample_id"].duplicated().any():
            raise ValueError(f"{compartment}: expected 127 unique patients")
        expected_groups = {"normal_weight": 52, "overweight": 57, "obese": 18}
        if cohort["bmi_group"].value_counts().to_dict() != expected_groups:
            raise ValueError(f"{compartment}: BMI cohort counts changed")

        comparison_columns = [
            "base_sample_id", "bmi", "bmi_z", "bmi_group", "age_z", "sex_centered"
        ]
        comparison = cohort[comparison_columns].sort_values("base_sample_id").reset_index(drop=True)
        if cohort_reference is None:
            cohort_reference = comparison
        else:
            if not cohort_reference[["base_sample_id", "bmi_group"]].equals(
                comparison[["base_sample_id", "bmi_group"]]
            ):
                raise ValueError("Patient IDs or BMI groups differ across compartments")
            for column in ("bmi", "bmi_z", "age_z", "sex_centered"):
                if not np.allclose(
                    cohort_reference[column], comparison[column], rtol=0, atol=1e-12
                ):
                    raise ValueError(f"Patient covariate {column} differs across compartments")

        local_tiers = tiers[tiers["compartment"].eq(compartment)].copy()
        if tier_column not in local_tiers.columns:
            raise KeyError(f"Missing tier column {tier_column}")
        keep_celltypes = sorted(
            local_tiers.loc[strict_bool_series(local_tiers[tier_column], tier_column), "CellType"].astype(str)
        )
        component_inventory = local_tiers.copy()
        component_inventory["requested_rule"] = tier_column
        component_inventory["included_in_this_run"] = component_inventory["CellType"].isin(
            keep_celltypes
        )
        component_inventory_parts.append(component_inventory)

        local = scores[
            scores["variant"].eq(SCORE_VARIANT)
            & scores["CellType"].isin(keep_celltypes)
        ].copy()
        if local.empty:
            raise ValueError(f"{compartment}: no scores for selected endpoint/components")
        candidate_features = set(local["feature"].astype(str))
        source_feature_count_before_mode += len(candidate_features)
        if FEATURE_MODE == "cluster_representative":
            representatives = choose_cluster_representatives(
                compartment, run_dir, candidate_features
            )
            local = local[local["feature"].isin(representatives)].copy()

        local.insert(0, "source_compartment", compartment)
        local["global_celltype"] = (
            local["source_compartment"].astype(str) + "||" + local["CellType"].astype(str)
        )
        local["global_feature"] = (
            local["source_compartment"].astype(str) + "||" + local["feature"].astype(str)
        )
        local = local.merge(
            cohort[comparison_columns],
            on="base_sample_id",
            how="inner",
            validate="many_to_one",
        )

        if INCLUDE_THETA:
            theta = patient_qc[
                patient_qc["CellType"].isin(keep_celltypes)
            ][["base_sample_id", "CellType", "theta"]].copy()
            if theta.duplicated(["base_sample_id", "CellType"]).any():
                raise ValueError(f"{compartment}: duplicate patient/celltype theta")
            theta["theta"] = pd.to_numeric(theta["theta"], errors="coerce")
            if theta["theta"].isna().any() or ((theta["theta"] < 0) | (theta["theta"] > 1)).any():
                raise ValueError(f"{compartment}: theta is missing or outside [0,1]")
            theta["theta_asin_sqrt"] = np.arcsin(np.sqrt(theta["theta"].clip(0, 1)))
            theta["theta_z"] = theta.groupby("CellType")["theta_asin_sqrt"].transform(
                lambda values: (values - values.mean()) / values.std(ddof=1)
            )
            local = local.merge(
                theta[["base_sample_id", "CellType", "theta", "theta_z"]],
                on=["base_sample_id", "CellType"],
                how="left",
                validate="many_to_one",
            )

        analysis_parts.append(local)
        source_hash_rows.extend(
            [
                {"compartment": compartment, "kind": "cohort", "path": str(cohort_path), "sha256": sha256_file(cohort_path)},
                {"compartment": compartment, "kind": "scores", "path": str(score_path), "sha256": sha256_file(score_path)},
                {"compartment": compartment, "kind": "patient_qc", "path": str(patient_qc_path), "sha256": sha256_file(patient_qc_path)},
            ]
        )

    analysis_data = pd.concat(analysis_parts, ignore_index=True)
    if analysis_data.duplicated(["base_sample_id", "global_feature"]).any():
        raise ValueError("Conflicting duplicate patient/feature rows")

    # Final feature standardization preserves the estimand: outcome SD per BMI SD.
    analysis_data["score"] = pd.to_numeric(analysis_data["score"], errors="coerce")
    feature_mean = analysis_data.groupby("global_feature")["score"].transform("mean")
    feature_sd = analysis_data.groupby("global_feature")["score"].transform("std")
    if feature_sd.isna().any() or (feature_sd <= 0).any():
        raise ValueError("At least one selected feature is constant/nonfinite")
    analysis_data["score_model"] = (analysis_data["score"] - feature_mean) / feature_sd
    analysis_data["bmi_standardized"] = analysis_data["bmi_z"]

    # Orthogonal quadratic sensitivity: no reuse of the linear BMI direction.
    patient_table_for_shape = cohort_reference.copy()
    shape_design = np.column_stack(
        [
            np.ones(len(patient_table_for_shape)),
            patient_table_for_shape[["bmi_z", "age_z", "sex_centered"]].to_numpy(float),
        ]
    )
    quadratic_raw = np.square(patient_table_for_shape["bmi_z"].to_numpy(float))
    quadratic_residual = quadratic_raw - shape_design @ (
        np.linalg.pinv(shape_design) @ quadratic_raw
    )
    quadratic_residual = (
        quadratic_residual - quadratic_residual.mean()
    ) / quadratic_residual.std(ddof=1)
    quadratic_map = dict(
        zip(patient_table_for_shape["base_sample_id"], quadratic_residual)
    )
    analysis_data["bmi_quadratic_orthogonal"] = analysis_data["base_sample_id"].map(
        quadratic_map
    )

    required_numeric = [
        "score_model", "bmi_standardized", "age_z", "sex_centered",
        "bmi_quadratic_orthogonal",
    ]
    if INCLUDE_THETA:
        required_numeric.append("theta_z")
    if not np.isfinite(analysis_data[required_numeric].to_numpy(float)).all():
        raise ValueError("Analysis table contains missing/nonfinite model values")

    feature_counts = analysis_data.groupby("global_feature")["base_sample_id"].nunique()
    if not feature_counts.eq(127).all():
        raise ValueError("Every modeled feature must contain all 127 patients")

    n_features = analysis_data["global_feature"].nunique()
    n_celltypes = analysis_data["global_celltype"].nunique()
    n_patients = analysis_data["base_sample_id"].nunique()
    expected = RUN_SPECS[RUN_SPEC]
    if source_feature_count_before_mode != expected["expected_features"]:
        raise ValueError(
            f"Expected {expected['expected_features']} source features before feature-mode "
            f"selection; found {source_feature_count_before_mode}"
        )
    if n_celltypes != expected["expected_celltypes"]:
        raise ValueError(
            f"Expected {expected['expected_celltypes']} celltypes; found {n_celltypes}"
        )
    if FEATURE_MODE == "all" and n_features != expected["expected_features"]:
        raise ValueError(
            f"Expected {expected['expected_features']} modeled features; found {n_features}"
        )
    if len(analysis_data) != n_features * n_patients:
        raise ValueError("Analysis table is not a complete patient-by-feature rectangle")

    analysis_data = analysis_data.sort_values(
        ["global_celltype", "global_feature", "base_sample_id"]
    ).reset_index(drop=True)

    analysis_path = OUTPUT_DIR / "analysis_input_long.csv.gz"
    atomic_csv(
        analysis_data,
        analysis_path,
        compression={"method": "gzip", "compresslevel": 6, "mtime": 0},
    )
    component_inventory = pd.concat(component_inventory_parts, ignore_index=True)
    atomic_csv(component_inventory, OUTPUT_DIR / "component_inventory.csv")
    atomic_csv(pd.DataFrame(source_hash_rows), OUTPUT_DIR / "source_input_hashes.csv")

    feature_inventory = analysis_data[
        [
            "source_compartment", "CellType", "Signature", "feature",
            "global_celltype", "global_feature",
        ]
    ].drop_duplicates()
    cluster_lookup = cluster_membership[
        cluster_membership["variant"].eq(CLUSTER_REFERENCE_VARIANT)
        & cluster_membership["correlation_kind"].eq(
            "residual_bmi_age_sex_theta"
        )
    ][["compartment", "feature", "cluster_id"]].drop_duplicates()
    if cluster_lookup.duplicated(["compartment", "feature"]).any():
        raise ValueError("A feature maps to more than one normalized correlation cluster")
    feature_inventory = feature_inventory.merge(
        cluster_lookup,
        left_on=["source_compartment", "feature"],
        right_on=["compartment", "feature"],
        how="left",
        validate="one_to_one",
    ).drop(columns="compartment")
    if feature_inventory["cluster_id"].isna().any():
        missing_cluster_features = feature_inventory.loc[
            feature_inventory["cluster_id"].isna(), "global_feature"
        ].tolist()
        raise ValueError(
            "Normalized correlation-cluster membership is missing for: "
            f"{missing_cluster_features[:10]}"
        )
    feature_inventory["global_cluster_id"] = (
        feature_inventory["source_compartment"].astype(str)
        + "||"
        + feature_inventory["cluster_id"].astype(str)
    )
    feature_inventory["cluster_size_in_run"] = feature_inventory.groupby(
        "global_cluster_id"
    )["global_feature"].transform("nunique")
    feature_inventory = feature_inventory.sort_values("global_feature")
    feature_inventory_path = OUTPUT_DIR / "feature_inventory.csv"
    atomic_csv(feature_inventory, feature_inventory_path)

    preflight_manifest = {
        "analysis_version": ANALYSIS_VERSION,
        "model_architecture": MODEL_ARCHITECTURE,
        "gate_version": GATE_VERSION,
        "notebook_release": NOTEBOOK_RELEASE,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "run_spec": RUN_SPEC,
        "role": RUN_SPECS[RUN_SPEC]["role"],
        "analysis_family": RUN_SPECS[RUN_SPEC]["analysis_family"],
        "profile": PROFILE,
        "score_variant": SCORE_VARIANT,
        "feature_mode": FEATURE_MODE,
        "bmi_shape": BMI_SHAPE,
        "include_theta": INCLUDE_THETA,
        "likelihood": LIKELIHOOD,
        "nuts_backend": NUTS_BACKEND,
        "seed": RUN_SEED,
        "pca_directory": str(PCA_DIR),
        "pca_manifest_sha256": sha256_file(pca_manifest_path),
        "combined_review_manifest_sha256": sha256_file(combined_manifest_path),
        "component_inclusion_tiers_sha256": sha256_file(tiers_path),
        "pca_stability_sha256": sha256_file(pca_stability_path),
        "pca_input_hashes_sha256": sha256_file(pca_hashes_path),
        "normalized_cluster_membership_sha256": sha256_file(cluster_membership_path),
        "feature_inventory_sha256": sha256_file(feature_inventory_path),
        "analysis_input_sha256": sha256_file(analysis_path),
        "n_patients": n_patients,
        "n_celltypes": n_celltypes,
        "n_features": n_features,
        "n_source_features_before_feature_mode": source_feature_count_before_mode,
        "n_correlation_clusters": int(
            feature_inventory["global_cluster_id"].nunique()
        ),
        "n_multifeature_correlation_clusters": int(
            feature_inventory.loc[
                feature_inventory["cluster_size_in_run"] > 1,
                "global_cluster_id",
            ].nunique()
        ),
        "n_observations": len(analysis_data),
        "cohort_counts": cohort_reference["bmi_group"].value_counts().to_dict(),
        "estimand": {
            "model_parameter": (
                "change in feature-standardized normalized signature score "
                "per one cohort BMI standard deviation"
            ),
            "bmi_sd_kg_m2": float(cohort_reference["bmi"].std(ddof=1)),
            "clinical_display_scale": (
                "change in normalized signature-score SD per 5 kg/m^2 higher BMI"
            ),
            "not_cohens_d": True,
        },
        "included_components": component_inventory.loc[
            component_inventory["included_in_this_run"],
            ["compartment", "CellType", "requested_rule"],
        ].to_dict("records"),
        "priors": {
            "baseline_sd": BASELINE_SD,
            "celltype_bmi_sd": CELLTYPE_BMI_SD,
            "feature_bmi_sd": FEATURE_BMI_SD,
            "celltype_covariate_sd": CELLTYPE_COVARIATE_SD,
            "feature_covariate_sd": FEATURE_COVARIATE_SD,
            "patient_global_sd": PATIENT_GLOBAL_SD,
            "log_sigma_location": LOG_SIGMA_LOCATION,
            "log_sigma_location_sd": LOG_SIGMA_LOCATION_SD,
            "celltype_log_sigma_sd": CELLTYPE_LOG_SIGMA_SD,
            "feature_log_sigma_sd": FEATURE_LOG_SIGMA_SD,
            "nu_minus_two_distribution": "Exponential",
            "nu_minus_two_rate": NU_MINUS_TWO_RATE,
            **({
                "celltype_quadratic_sd": CELLTYPE_QUADRATIC_SD,
                "feature_quadratic_sd": FEATURE_QUADRATIC_SD,
            } if BMI_SHAPE == "linear_quadratic" else {}),
            **({
                "celltype_theta_sd": CELLTYPE_THETA_SD,
                "feature_theta_sd": FEATURE_THETA_SD,
            } if INCLUDE_THETA else {}),
        },
        "rope_grid": list(ROPE_GRID),
        "primary_rope": PRIMARY_ROPE,
        "q_level": Q_LEVEL,
        "patient_effect_structure": {
            "global_patient_effect": "centered_fixed_residual_basis",
            "global_patient_loading": 1.0,
            "patient_by_celltype_effect": False,
            "estimated_patient_loadings": False,
            "patient_global_sigma_interpretation": (
                "scale in the normalized residual basis; its square is the "
                "mean marginal patient-effect variance, not an unrestricted "
                "raw between-patient variance"
            ),
        },
        "student_t_nu_scope": "one_nu_per_celltype_within_separately_fitted_compartment",
        "student_t_sigma_interpretation": (
            "sigma_feature is the Student-t scale parameter; conditional SD is "
            "sigma_feature * sqrt(nu_celltype / (nu_celltype - 2))"
        ),
        "versions": {
            "python": platform.python_version(),
            "pymc": pm.__version__,
            "arviz": az.__version__,
            "matplotlib": mpl.__version__,
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "nutpie": runtime_nutpie_version,
        },
    }
    atomic_text(
        json.dumps(preflight_manifest, indent=2, default=str),
        OUTPUT_DIR / "preflight_manifest.json",
    )

    print(json.dumps(preflight_manifest, indent=2, default=str))
    print("\nPCA stability rows used for design decisions")
    display(
        pca_stability[
            pca_stability["compartment"].isin(
                [item[0] for item in RUN_SPECS[RUN_SPEC]["components"]]
            )
        ]
    )


In [ ]:
# 5. Build and lock the v2.2 cell-type-nu compartment model
if RUN_PRIMARY_AGGREGATION:
    model = None
    model_arrays = None
    MODEL_FINGERPRINT = None
    ANALYSIS_LOCK_SHA256 = None
    print('Aggregation stage: current-run model construction is intentionally skipped.')
else:
    # 5. Build the v2.2 global-patient fixed-loading, cell-type-nu model

    def prepare_model_arrays(frame: pd.DataFrame):
        data = frame.sort_values(
            ["global_celltype", "global_feature", "base_sample_id"]
        ).reset_index(drop=True).copy()
        features = sorted(data["global_feature"].unique())
        celltypes = sorted(data["global_celltype"].unique())
        patients = sorted(data["base_sample_id"].unique())
        feature_index = {name: index for index, name in enumerate(features)}
        celltype_index = {name: index for index, name in enumerate(celltypes)}
        patient_index = {name: index for index, name in enumerate(patients)}
        data["feature_idx"] = data["global_feature"].map(feature_index)
        data["celltype_idx"] = data["global_celltype"].map(celltype_index)
        data["patient_idx"] = data["base_sample_id"].map(patient_index)

        feature_mapping = (
            data[["feature_idx", "celltype_idx"]]
            .drop_duplicates()
            .sort_values("feature_idx")
        )
        if len(feature_mapping) != len(features) or feature_mapping["feature_idx"].duplicated().any():
            raise ValueError("Every feature must map to exactly one cell type")
        feature_to_celltype = feature_mapping["celltype_idx"].to_numpy(int)

        patient_predictors = ["bmi_standardized", "age_z", "sex_centered"]
        if BMI_SHAPE == "linear_quadratic":
            patient_predictors.append("bmi_quadratic_orthogonal")
        consistency = data.groupby("patient_idx")[patient_predictors].nunique(dropna=False)
        if (consistency > 1).to_numpy().any():
            raise ValueError("A patient-level predictor varies across that patient's features")
        patient_table = (
            data[["patient_idx", *patient_predictors]]
            .drop_duplicates("patient_idx")
            .sort_values("patient_idx")
            .reset_index(drop=True)
        )
        if not np.array_equal(patient_table["patient_idx"], np.arange(len(patients))):
            raise ValueError("Patient ordering is inconsistent")

        # Build the complete patient-level fixed-effect span used by this run.
        # The optional theta adjustment is feature/cell-type specific, so all
        # patient-by-cell-type theta vectors must enter the nuisance span.
        patient_design_columns = ["intercept", *patient_predictors]
        base_patient_design = np.column_stack(
            [np.ones(len(patients)), patient_table[patient_predictors].to_numpy(float)]
        )
        if not np.isfinite(base_patient_design).all():
            raise ValueError("Base patient fixed-effect design is nonfinite")
        base_rank = int(np.linalg.matrix_rank(base_patient_design))
        if base_rank != base_patient_design.shape[1]:
            raise ValueError("Base patient fixed-effect design is rank deficient")

        patient_design_parts = [base_patient_design]
        if INCLUDE_THETA:
            theta_consistency = data.groupby(
                ["patient_idx", "celltype_idx"]
            )["theta_z"].nunique(dropna=False)
            if (theta_consistency > 1).any():
                raise ValueError(
                    "theta_z varies across features within a patient/cell-type pair"
                )
            theta_table = (
                data[["patient_idx", "celltype_idx", "theta_z"]]
                .drop_duplicates(["patient_idx", "celltype_idx"])
                .pivot(index="patient_idx", columns="celltype_idx", values="theta_z")
                .reindex(index=np.arange(len(patients)), columns=np.arange(len(celltypes)))
            )
            if theta_table.isna().to_numpy().any():
                raise ValueError("Theta design lacks a patient/cell-type value")
            theta_design = theta_table.to_numpy(float)
            if not np.isfinite(theta_design).all():
                raise ValueError("Theta patient fixed-effect design is nonfinite")
            patient_design_parts.append(theta_design)
            patient_design_columns.extend(
                [f"theta_z__{name}" for name in celltypes]
            )

        patient_design = np.column_stack(patient_design_parts)
        u, singular_values, _ = np.linalg.svd(patient_design, full_matrices=True)
        tolerance = max(patient_design.shape) * np.finfo(float).eps * singular_values.max()
        design_rank = int((singular_values > tolerance).sum())
        if design_rank < base_rank:
            raise AssertionError("Complete patient design lost base fixed-effect rank")
        if design_rank >= len(patients):
            raise ValueError("Patient design leaves no residual random-effect dimension")
        patient_basis = u[:, design_rank:]
        # The scaling makes the mean marginal variance of the patient-effect vector
        # equal to patient_global_sigma**2; individual variances depend on leverage.
        patient_basis *= np.sqrt(len(patients) / patient_basis.shape[1])
        orthogonality_error = float(np.abs(patient_design.T @ patient_basis).max())
        if orthogonality_error > 1e-10:
            raise AssertionError(f"Patient basis is not orthogonal: {orthogonality_error}")

        feature_count_by_celltype = np.bincount(
            feature_to_celltype, minlength=len(celltypes)
        ).astype(int)

        return {
            "data": data,
            "features": features,
            "celltypes": celltypes,
            "patients": patients,
            "feature_to_celltype": feature_to_celltype,
            "patient_basis": patient_basis,
            "patient_design_rank": design_rank,
            "patient_design_column_count": patient_design.shape[1],
            "patient_design_columns": patient_design_columns,
            "patient_base_design_rank": base_rank,
            "patient_orthogonality_error": orthogonality_error,
            "feature_count_by_celltype": feature_count_by_celltype,
        }

    def build_normalized_state_model(frame: pd.DataFrame):
        arrays = prepare_model_arrays(frame)
        data = arrays["data"]
        features = arrays["features"]
        celltypes = arrays["celltypes"]
        patients = arrays["patients"]
        covariates = ["age_z", "sex_centered"]

        coords = {
            "obs_id": np.arange(len(data)),
            "feature": features,
            "celltype": celltypes,
            "patient": patients,
            "patient_re_dimension": np.arange(arrays["patient_basis"].shape[1]),
            "covariate": covariates,
        }

        with pm.Model(coords=coords) as model:
            feature_idx = pm.Data("feature_idx", data["feature_idx"].to_numpy(int), dims="obs_id")
            patient_idx = pm.Data("patient_idx", data["patient_idx"].to_numpy(int), dims="obs_id")
            celltype_idx = pm.Data("celltype_idx", data["celltype_idx"].to_numpy(int), dims="obs_id")
            feature_to_celltype = pm.Data(
                "feature_to_celltype", arrays["feature_to_celltype"], dims="feature"
            )
            bmi_z = pm.Data("bmi_z", data["bmi_standardized"].to_numpy(float), dims="obs_id")
            covariate_matrix = pm.Data(
                "covariate_matrix", data[covariates].to_numpy(float),
                dims=("obs_id", "covariate"),
            )
            y = pm.Data("y", data["score_model"].to_numpy(float), dims="obs_id")
            patient_residual_basis = pm.Data(
                "patient_residual_basis", arrays["patient_basis"],
                dims=("patient", "patient_re_dimension"),
            )

            feature_baseline = pm.Normal(
                "feature_baseline", 0.0, BASELINE_SD, dims="feature"
            )

            # Feature BMI effects partially pool through their cell type.
            celltype_bmi_slope = pm.Normal(
                "celltype_bmi_slope", 0.0, CELLTYPE_BMI_SD, dims="celltype"
            )
            feature_sigma_bmi = pm.HalfNormal(
                "feature_sigma_bmi", FEATURE_BMI_SD, dims="celltype"
            )
            feature_bmi_raw = pm.Normal(
                "feature_bmi_raw", 0.0, 1.0, dims="feature"
            )
            feature_bmi_slope = pm.Deterministic(
                "feature_bmi_slope",
                celltype_bmi_slope[feature_to_celltype]
                + feature_bmi_raw * feature_sigma_bmi[feature_to_celltype],
                dims="feature",
            )

            # Age and sex are feature-specific with cell-type partial pooling;
            
            celltype_covariate_slope = pm.Normal(
                "celltype_covariate_slope", 0.0, CELLTYPE_COVARIATE_SD,
                dims=("covariate", "celltype"),
            )
            feature_sigma_covariate = pm.HalfNormal(
                "feature_sigma_covariate", FEATURE_COVARIATE_SD,
                dims=("covariate", "celltype"),
            )
            feature_covariate_raw = pm.Normal(
                "feature_covariate_raw", 0.0, 1.0,
                dims=("covariate", "feature"),
            )
            feature_covariate_slope = pm.Deterministic(
                "feature_covariate_slope",
                celltype_covariate_slope[:, feature_to_celltype]
                + feature_covariate_raw
                * feature_sigma_covariate[:, feature_to_celltype],
                dims=("covariate", "feature"),
            )

            # Orthogonal global patient effect.
            patient_global_sigma = pm.HalfNormal(
                "patient_global_sigma", PATIENT_GLOBAL_SD
            )

            patient_global_coefficient = pm.Normal(
                "patient_global_coefficient",
                mu=0.0,
                sigma=patient_global_sigma,
                dims="patient_re_dimension",
            )

            patient_global_effect = pm.Deterministic(
                "patient_global_effect",
                pt.dot(patient_residual_basis, patient_global_coefficient),
                dims="patient",
            )

            mu = (
                feature_baseline[feature_idx]
                + feature_bmi_slope[feature_idx] * bmi_z
                + patient_global_effect[patient_idx]
            )
            feature_covariate_at_obs = feature_covariate_slope[:, feature_idx].T
            mu = mu + pt.sum(
                covariate_matrix * feature_covariate_at_obs, axis=1
            )

            if BMI_SHAPE == "linear_quadratic":
                bmi_quadratic = pm.Data(
                    "bmi_quadratic",
                    data["bmi_quadratic_orthogonal"].to_numpy(float),
                    dims="obs_id",
                )
                celltype_bmi_quadratic = pm.Normal(
                    "celltype_bmi_quadratic", 0.0, CELLTYPE_QUADRATIC_SD,
                    dims="celltype",
                )
                feature_sigma_quadratic = pm.HalfNormal(
                    "feature_sigma_quadratic", FEATURE_QUADRATIC_SD,
                    dims="celltype",
                )
                feature_quadratic_raw = pm.Normal(
                    "feature_quadratic_raw", 0.0, 1.0, dims="feature"
                )
                feature_bmi_quadratic = pm.Deterministic(
                    "feature_bmi_quadratic",
                    celltype_bmi_quadratic[feature_to_celltype]
                    + feature_quadratic_raw
                    * feature_sigma_quadratic[feature_to_celltype],
                    dims="feature",
                )
                mu = mu + feature_bmi_quadratic[feature_idx] * bmi_quadratic

            if INCLUDE_THETA:
                theta_z = pm.Data(
                    "theta_z", data["theta_z"].to_numpy(float), dims="obs_id"
                )
                celltype_theta_slope = pm.Normal(
                    "celltype_theta_slope", 0.0, CELLTYPE_THETA_SD,
                    dims="celltype",
                )
                feature_sigma_theta = pm.HalfNormal(
                    "feature_sigma_theta", FEATURE_THETA_SD, dims="celltype"
                )
                feature_theta_raw = pm.Normal(
                    "feature_theta_raw", 0.0, 1.0, dims="feature"
                )
                feature_theta_slope = pm.Deterministic(
                    "feature_theta_slope",
                    celltype_theta_slope[feature_to_celltype]
                    + feature_theta_raw * feature_sigma_theta[feature_to_celltype],
                    dims="feature",
                )
                mu = mu + feature_theta_slope[feature_idx] * theta_z

            # Cell-type and feature-specific residual scales. The hierarchy can
            # represent both directions of the dispersion error seen in the v1 PPC.
            log_sigma_location = pm.Normal(
                "log_sigma_location", LOG_SIGMA_LOCATION, LOG_SIGMA_LOCATION_SD
            )
            log_sigma_celltype_location_sd = pm.HalfNormal(
                "log_sigma_celltype_location_sd", CELLTYPE_LOG_SIGMA_SD
            )
            log_sigma_celltype_raw = pm.Normal(
                "log_sigma_celltype_raw", 0.0, 1.0, dims="celltype"
            )
            log_sigma_celltype_location = pm.Deterministic(
                "log_sigma_celltype_location",
                log_sigma_location
                + log_sigma_celltype_location_sd * log_sigma_celltype_raw,
                dims="celltype",
            )
            log_sigma_feature_scale = pm.HalfNormal(
                "log_sigma_feature_scale", FEATURE_LOG_SIGMA_SD, dims="celltype"
            )
            log_sigma_raw = pm.Normal(
                "log_sigma_raw", 0.0, 1.0, dims="feature"
            )
            sigma_feature = pm.Deterministic(
                "sigma_feature",
                pt.exp(
                    log_sigma_celltype_location[feature_to_celltype]
                    + log_sigma_feature_scale[feature_to_celltype] * log_sigma_raw
                ),
                dims="feature",
            )

            # One regularized tail parameter per cell type. This is the only
            # structural change from v2.1c and directly addresses the
            # immune-specific tail mismatch in corrected PPC v2.
            nu_minus_two = pm.Exponential(
                "nu_minus_two", lam=NU_MINUS_TWO_RATE, dims="celltype"
            )
            nu = pm.Deterministic(
                "nu", nu_minus_two + 2.0, dims="celltype"
            )
            pm.StudentT(
                "y_obs",
                nu=nu[celltype_idx],
                mu=mu,
                sigma=sigma_feature[feature_idx],
                observed=y,
                dims="obs_id",
            )

        return model, arrays

    model, model_arrays = build_normalized_state_model(analysis_data)
    initial_logp = float(model.compile_logp()(model.initial_point()))
    if not np.isfinite(initial_logp):
        raise ValueError(f"Initial model log-probability is not finite: {initial_logp}")

    EXPECTED_BASE_FREE_RVS = {
        "feature_baseline", "celltype_bmi_slope", "feature_sigma_bmi",
        "feature_bmi_raw", "celltype_covariate_slope", "feature_sigma_covariate",
        "feature_covariate_raw", "patient_global_sigma", "patient_global_coefficient",
        "log_sigma_location", "log_sigma_celltype_location_sd",
        "log_sigma_celltype_raw", "log_sigma_feature_scale", "log_sigma_raw",
        "nu_minus_two",
    }
    expected_free_rvs = set(EXPECTED_BASE_FREE_RVS)
    if BMI_SHAPE == "linear_quadratic":
        expected_free_rvs |= {
            "celltype_bmi_quadratic", "feature_sigma_quadratic", "feature_quadratic_raw"
        }
    if INCLUDE_THETA:
        expected_free_rvs |= {
            "celltype_theta_slope", "feature_sigma_theta", "feature_theta_raw"
        }
    actual_free_rvs = {variable.name for variable in model.free_RVs}
    if actual_free_rvs != expected_free_rvs:
        raise AssertionError(
            f"Free-RV contract changed; missing={sorted(expected_free_rvs-actual_free_rvs)}, "
            f"extra={sorted(actual_free_rvs-expected_free_rvs)}"
        )
    forbidden_v2_names = {
        "patient_celltype_sigma", "patient_celltype_raw", "patient_celltype_effect",
        "global_loading_sd", "global_loading_raw", "global_loading",
        "celltype_loading_sd", "celltype_loading_raw", "celltype_loading", "patient_global_raw",
    }
    present_forbidden = sorted(forbidden_v2_names & set(model.named_vars))
    if present_forbidden:
        raise AssertionError(f"Forbidden failed-v2 variables are present: {present_forbidden}")
    if model["nu"].ndim != 1:
        raise AssertionError("v2.2 requires one Student-t nu per cell type")

    required_derived_diagnostics = [
        "feature_bmi_slope", "feature_covariate_slope", "patient_global_effect",
        "log_sigma_celltype_location", "sigma_feature", "nu",
    ]
    if BMI_SHAPE == "linear_quadratic":
        required_derived_diagnostics.append("feature_bmi_quadratic")
    if INCLUDE_THETA:
        required_derived_diagnostics.append("feature_theta_slope")

    model_structure = {
        "analysis_version": ANALYSIS_VERSION,
        "model_architecture": MODEL_ARCHITECTURE,
        "initial_logp": initial_logp,
        "n_free_rv_families": len(model.free_RVs),
        "n_unconstrained_scalar_coordinates": int(sum(
            np.size(value) for value in model.initial_point().values()
        )),
        "free_rvs": sorted(actual_free_rvs),
        "required_derived_diagnostics": sorted(required_derived_diagnostics),
        "patient_design_rank": model_arrays["patient_design_rank"],
        "patient_design_column_count": model_arrays["patient_design_column_count"],
        "patient_design_columns": model_arrays["patient_design_columns"],
        "patient_base_design_rank": model_arrays["patient_base_design_rank"],
        "patient_re_dimensions": model_arrays["patient_basis"].shape[1],
        "patient_orthogonality_max_abs_error": model_arrays["patient_orthogonality_error"],
        "patient_global_sigma_interpretation": (
            "normalized residual-basis scale: its square equals the mean marginal "
            "patient-effect variance; individual variances depend on design leverage"
        ),
        "global_patient_loading": 1.0,
        "patient_by_celltype_effect": False,
        "nu_scope": "celltype_within_separate_compartment",
        "sigma_feature_interpretation": (
            "Student-t scale parameter; conditional residual SD equals "
            "sigma_feature * sqrt(nu_celltype / (nu_celltype - 2))"
        ),
        "feature_count_by_celltype": dict(zip(
            model_arrays["celltypes"], model_arrays["feature_count_by_celltype"].tolist()
        )),
    }
    atomic_json(model_structure, OUTPUT_DIR / "model_structure.json")

    model_fingerprint_payload = {
        "analysis_version": ANALYSIS_VERSION,
        "model_architecture": MODEL_ARCHITECTURE,
        "gate_version": GATE_VERSION,
        "ppc_version": PPC_VERSION,
        "notebook_release_sha256": NOTEBOOK_RELEASE["sha256"],
        "run_spec": RUN_SPEC,
        "score_variant": SCORE_VARIANT,
        "feature_mode": FEATURE_MODE,
        "bmi_shape": BMI_SHAPE,
        "include_theta": INCLUDE_THETA,
        "likelihood": LIKELIHOOD,
        "nuts_backend": NUTS_BACKEND,
        "analysis_input_sha256": preflight_manifest["analysis_input_sha256"],
        "pca_manifest_sha256": preflight_manifest["pca_manifest_sha256"],
        "combined_review_manifest_sha256": preflight_manifest["combined_review_manifest_sha256"],
        "component_inclusion_tiers_sha256": preflight_manifest["component_inclusion_tiers_sha256"],
        "pca_stability_sha256": preflight_manifest["pca_stability_sha256"],
        "pca_input_hashes_sha256": preflight_manifest["pca_input_hashes_sha256"],
        "normalized_cluster_membership_sha256": preflight_manifest["normalized_cluster_membership_sha256"],
        "feature_inventory_sha256": preflight_manifest["feature_inventory_sha256"],
        "priors": preflight_manifest["priors"],
        "free_rvs": sorted(actual_free_rvs),
        "required_derived_diagnostics": sorted(required_derived_diagnostics),
    }
    MODEL_FINGERPRINT = sha256_json(model_fingerprint_payload)
    analysis_lock = {
        **model_fingerprint_payload,
        "model_fingerprint": MODEL_FINGERPRINT,
        "profile": PROFILE,
        "sampler_profile": PROFILES[PROFILE],
        "run_seed": RUN_SEED,
        "rope_grid": list(ROPE_GRID),
        "primary_rope": PRIMARY_ROPE,
        "q_level": Q_LEVEL,
    }
    ANALYSIS_LOCK_PATH = OUTPUT_DIR / "analysis_lock.json"
    canonical_lock = json.dumps(analysis_lock, indent=2, sort_keys=True, default=str)
    if ANALYSIS_LOCK_PATH.exists():
        existing_lock = ANALYSIS_LOCK_PATH.read_text(encoding="utf-8")
        if existing_lock != canonical_lock:
            raise RuntimeError(
                "The write-once analysis lock differs from the current code/input/config. "
                "Use a new RUN_SPEC/profile or restore the locked notebook."
            )
    else:
        atomic_text(canonical_lock, ANALYSIS_LOCK_PATH)
    ANALYSIS_LOCK_SHA256 = sha256_file(ANALYSIS_LOCK_PATH)

    print(json.dumps(model_structure, indent=2, default=str))
    print("Model fingerprint:", MODEL_FINGERPRINT)
    print("Analysis-lock SHA256:", ANALYSIS_LOCK_SHA256)

    if RUN_PRIOR_CHECK:
        prior_parameter_variables = [
            "feature_bmi_slope", "celltype_bmi_slope", "feature_sigma_bmi",
            "feature_covariate_slope", "celltype_covariate_slope",
            "feature_sigma_covariate", "patient_global_sigma",
            "patient_global_effect", "log_sigma_celltype_location_sd",
            "log_sigma_celltype_location", "log_sigma_feature_scale",
            "sigma_feature", "nu",
        ]
        if BMI_SHAPE == "linear_quadratic":
            prior_parameter_variables.append("feature_bmi_quadratic")
        if INCLUDE_THETA:
            prior_parameter_variables.append("feature_theta_slope")
        with model:
            prior_idata = pm.sample_prior_predictive(
                draws=PRIOR_DRAWS,
                var_names=prior_parameter_variables + ["y_obs"],
                random_seed=RUN_SEED,
            )
        prior_rows = []
        for variable in prior_parameter_variables:
            values = np.asarray(prior_idata.prior[variable]).reshape(-1)
            prior_rows.append({
                "variable": variable,
                "q005": float(np.quantile(values, 0.005)),
                "q025": float(np.quantile(values, 0.025)),
                "median": float(np.quantile(values, 0.5)),
                "q975": float(np.quantile(values, 0.975)),
                "q995": float(np.quantile(values, 0.995)),
            })
        prior_summary = pd.DataFrame(prior_rows)
        atomic_csv(prior_summary, OUTPUT_DIR / "prior_predictive_parameter_summary.csv")
        prior_y = np.asarray(prior_idata.prior_predictive["y_obs"]).reshape(-1)
        prior_outcome_summary = {
            "prior_draws": PRIOR_DRAWS,
            "outcome_scale": "feature-standardized normalized signature score",
            **{name: float(np.quantile(prior_y, q)) for name, q in (
                ("q005", 0.005), ("q025", 0.025), ("median", 0.5),
                ("q975", 0.975), ("q995", 0.995),
            )},
            "fraction_abs_gt_3": float(np.mean(np.abs(prior_y) > 3)),
            "fraction_abs_gt_5": float(np.mean(np.abs(prior_y) > 5)),
            "fraction_abs_gt_10": float(np.mean(np.abs(prior_y) > 10)),
        }
        atomic_json(prior_outcome_summary, OUTPUT_DIR / "prior_predictive_outcome_summary.json")
        display(prior_summary)
        print(json.dumps(prior_outcome_summary, indent=2))
        del prior_y, prior_idata


In [ ]:
# 6. Sample/load atomically and run profile-specific diagnostics
def netcdf_safe_attr(value):
    if value is None: return "null"
    if isinstance(value, (bool, np.bool_)): return int(value)
    if isinstance(value, np.generic): return value.item()
    if isinstance(value, (str, bytes, int, float)): return value
    if isinstance(value, np.ndarray) and value.dtype != object: return value
    return json.dumps(value, default=str, sort_keys=True)

def sanitize_idata_attrs(idata):
    owners = [idata]
    for group_name in idata.groups():
        dataset = getattr(idata, group_name)
        owners.append(dataset)
        owners.extend(dataset.variables.values())
    for owner in owners:
        attrs = getattr(owner, "attrs", None)
        if attrs is not None:
            for key, value in list(attrs.items()):
                attrs[key] = netcdf_safe_attr(value)
    return idata

def save_idata_atomic(idata, path: Path) -> Path:
    temporary = path.with_name(f".{path.name}.tmp")
    if temporary.exists(): temporary.unlink()
    sanitize_idata_attrs(idata)
    try:
        idata.to_netcdf(temporary)
        temporary.replace(path)
    finally:
        if temporary.exists(): temporary.unlink()
    return path

def require_profile_promotion() -> dict | None:
    predecessor = PROFILE_PREDECESSOR[PROFILE]
    if predecessor is None:
        return None
    predecessor_dir = run_directory_for(RUN_SPEC, predecessor)
    diagnostics_path = predecessor_dir / "run_diagnostics.json"
    trace_path = predecessor_dir / "posterior_trace.nc"
    lock_path = predecessor_dir / "analysis_lock.json"
    required = [diagnostics_path, trace_path, lock_path]
    if PROFILE in {"publication", "publication_long"}:
        required.append(predecessor_dir / "ppc_adequacy.json")
    if PROFILE == "publication_long":
        required.append(predecessor_dir / "reveal_precision_gate.json")
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError(f"{PROFILE} promotion requires: {missing}")
    previous = json.loads(diagnostics_path.read_text(encoding="utf-8"))
    checks = {
        "profile": previous.get("profile") == predecessor,
        "gate": previous.get("current_profile_gate_pass") is True,
        "architecture": previous.get("model_architecture") == MODEL_ARCHITECTURE,
        "fingerprint": previous.get("model_fingerprint") == MODEL_FINGERPRINT,
        "input": previous.get("analysis_input_sha256") == preflight_manifest["analysis_input_sha256"],
        "trace_hash": previous.get("posterior_trace_sha256") == sha256_file(trace_path),
        "lock_hash": previous.get("analysis_lock_sha256") == sha256_file(lock_path),
        "verified_notebook_release": previous.get("notebook_release_status") == "verified",
    }
    evidence = {
        "profile": predecessor,
        "directory": str(predecessor_dir),
        "diagnostics_sha256": sha256_file(diagnostics_path),
        "trace_sha256": sha256_file(trace_path),
        "analysis_lock_sha256": sha256_file(lock_path),
    }
    if PROFILE in {"publication", "publication_long"}:
        ppc_path = predecessor_dir / "ppc_adequacy.json"
        ppc = json.loads(ppc_path.read_text(encoding="utf-8"))
        checks.update({
            "predecessor_ppc_profile": ppc.get("profile") == predecessor,
            "predecessor_ppc_pass": ppc.get("ppc_adequacy_pass") is True,
            "predecessor_ppc_fingerprint": ppc.get("model_fingerprint") == MODEL_FINGERPRINT,
            "predecessor_ppc_trace": ppc.get("posterior_trace_sha256") == sha256_file(trace_path),
            "predecessor_ppc_lock": ppc.get("analysis_lock_sha256") == sha256_file(lock_path),
            "predecessor_ppc_diagnostics": ppc.get("diagnostics_sha256") == sha256_file(diagnostics_path),
            "predecessor_ppc_notebook_verified": ppc.get("notebook_release_status") == "verified",
        })
        evidence["ppc_adequacy_sha256"] = sha256_file(ppc_path)
    if PROFILE == "publication_long":
        precision_path = predecessor_dir / "reveal_precision_gate.json"
        precision = json.loads(precision_path.read_text(encoding="utf-8"))
        local_precision_failed = bool(
            precision.get("selection_status") == "withheld_precision_failed"
            and precision.get("all_displayed_ropes_pass") is False
        )
        local_precision_passed = bool(
            precision.get("selection_status") == "precision_pass"
            and precision.get("all_displayed_ropes_pass") is True
        )
        global_gate_path = (
            MODEL_ROOT / "global_primary_bfdr_v2p2" / "global_primary_precision_gate.json"
        )
        global_precision_failed = False
        if global_gate_path.exists() and ACTIVE_SPEC["role"] == "primary":
            global_gate = json.loads(global_gate_path.read_text(encoding="utf-8"))
            source_matches = [
                source for source in global_gate.get("primary_sources", [])
                if source.get("run_spec") == RUN_SPEC
            ]
            if len(source_matches) == 1:
                source = source_matches[0]
                global_precision_failed = bool(
                    global_gate.get("model_architecture") == MODEL_ARCHITECTURE
                    and global_gate.get("selection_status") == "withheld_precision_failed"
                    and global_gate.get("all_displayed_ropes_pass") is False
                    and source.get("profile") == "publication"
                    and Path(source.get("directory", "")).resolve() == predecessor_dir.resolve()
                    and source.get("analysis_lock_sha256") == sha256_file(lock_path)
                    and source.get("trace_sha256") == sha256_file(trace_path)
                    and source.get("diagnostics_sha256") == sha256_file(diagnostics_path)
                    and source.get("ppc_adequacy_sha256") == sha256_file(
                        predecessor_dir / "ppc_adequacy.json"
                    )
                    and source.get("precision_gate_sha256") == sha256_file(precision_path)
                    and (predecessor_dir / "selection_reveal_manifest.json").exists()
                    and source.get("reveal_manifest_sha256") == sha256_file(
                        predecessor_dir / "selection_reveal_manifest.json"
                    )
                    and (predecessor_dir / "revealed_feature_bmi_results.csv").exists()
                    and source.get("results_sha256") == sha256_file(
                        predecessor_dir / "revealed_feature_bmi_results.csv"
                    )
                )
        checks.update({
            "publication_precision_record_valid": local_precision_failed or local_precision_passed,
            "publication_long_has_prespecified_trigger": local_precision_failed or global_precision_failed,
            "publication_precision_architecture": precision.get("model_architecture") == MODEL_ARCHITECTURE,
            "publication_precision_fingerprint": precision.get("model_fingerprint") == MODEL_FINGERPRINT,
            "publication_precision_trace": precision.get("posterior_trace_sha256") == sha256_file(trace_path),
            "publication_precision_lock": precision.get("analysis_lock_sha256") == sha256_file(lock_path),
            "publication_precision_diagnostics": precision.get("diagnostics_sha256") == sha256_file(diagnostics_path),
            "publication_precision_ppc": precision.get("ppc_adequacy_sha256") == sha256_file(
                predecessor_dir / "ppc_adequacy.json"
            ),
        })
        evidence["publication_precision_gate_sha256"] = sha256_file(precision_path)
        evidence["publication_long_trigger"] = (
            "local_rope_bfdr_precision_failure"
            if local_precision_failed else "global_primary_bfdr_precision_failure"
        )
        if global_precision_failed:
            evidence["failed_global_precision_gate_sha256"] = sha256_file(global_gate_path)
    failed = [name for name, passed in checks.items() if not passed]
    if failed:
        raise RuntimeError(f"{PROFILE} promotion blocked; failed checks: {failed}")
    return evidence

TRACE_PATH = OUTPUT_DIR / "posterior_trace.nc"
promotion_evidence = None
if RUN_MODEL:
    if TRACE_PATH.exists():
        raise FileExistsError(
            f"{TRACE_PATH} exists. Load it with another stage; never overwrite a trace."
        )
    promotion_evidence = None
    sampler = PROFILES[PROFILE]
    sample_options = {
        "tune": sampler["tune"], "draws": sampler["draws"],
        "chains": sampler["chains"], "cores": sampler["chains"],
        "target_accept": sampler["target_accept"], "nuts_sampler": NUTS_BACKEND,
        "random_seed": RUN_SEED, "return_inferencedata": True,
    }
    if NUTS_BACKEND == "nutpie":
        sample_options["nuts_sampler_kwargs"] = {"maxdepth": MAX_TREE_DEPTH}
    elif NUTS_BACKEND == "pymc":
        sample_options["idata_kwargs"] = {"log_likelihood": False}
    with model:
        idata = pm.sample(**sample_options)
    trace_lock_attrs = {
        "analysis_lock_sha256": ANALYSIS_LOCK_SHA256,
        "model_fingerprint": MODEL_FINGERPRINT,
        "notebook_release_sha256": NOTEBOOK_RELEASE["sha256"],
        "model_architecture": MODEL_ARCHITECTURE,
        "promotion_evidence": json.dumps(promotion_evidence, sort_keys=True),
    }
    for key, value in trace_lock_attrs.items():
        idata.attrs[key] = value
        idata.posterior.attrs[key] = value
    save_idata_atomic(idata, TRACE_PATH)
elif WORKFLOW_STAGE in LOAD_TRACE_STAGES:
    if not TRACE_PATH.exists():
        raise FileNotFoundError(f"No trace exists for this exact run: {TRACE_PATH}")
    idata = az.from_netcdf(TRACE_PATH)
    expected_attrs = {
        "analysis_lock_sha256": ANALYSIS_LOCK_SHA256,
        "model_fingerprint": MODEL_FINGERPRINT,
        "notebook_release_sha256": NOTEBOOK_RELEASE["sha256"],
        "model_architecture": MODEL_ARCHITECTURE,
    }
    observed_attrs = {
        key: idata.attrs.get(key, idata.posterior.attrs.get(key))
        for key in expected_attrs
    }
    attr_failures = {
        key: {"expected": expected, "observed": observed_attrs.get(key)}
        for key, expected in expected_attrs.items()
        if observed_attrs.get(key) != expected
    }
    if attr_failures:
        raise RuntimeError(f"Trace/analysis-lock mismatch: {attr_failures}")
    stored_promotion = idata.attrs.get(
        "promotion_evidence", idata.posterior.attrs.get("promotion_evidence", "null")
    )
    promotion_evidence = json.loads(stored_promotion)
    print("Loaded locked trace:", TRACE_PATH)
else:
    idata = None
    print("No trace loaded for this workflow stage.")

def parameter_family(parameter: str) -> str:
    return re.sub(r"\[.*$", "", str(parameter))

if idata is not None:
    posterior_names = set(idata.posterior.data_vars)
    free_names = [variable.name for variable in model.free_RVs]
    missing_free = sorted(set(free_names) - posterior_names)
    missing_derived = sorted(set(required_derived_diagnostics) - posterior_names)
    if missing_free:
        raise RuntimeError(f"Trace lacks free variables: {missing_free}")
    diagnostic_names = sorted(set(free_names + required_derived_diagnostics) & posterior_names)
    full_diagnostics = az.summary(
        idata, var_names=diagnostic_names, round_to="none", kind="diagnostics"
    ).reset_index(names="parameter")
    full_diagnostics["parameter_family"] = full_diagnostics["parameter"].map(parameter_family)
    atomic_csv(full_diagnostics, OUTPUT_DIR / "full_diagnostics.csv")

    diagnostic_columns = ["r_hat", "ess_bulk", "ess_tail"]
    diagnostic_values_finite = bool(np.isfinite(
        full_diagnostics[diagnostic_columns].to_numpy(float)
    ).all())
    available_sample_stats = (
        set(idata.sample_stats.data_vars) if hasattr(idata, "sample_stats") else set()
    )
    required_sample_stats = {"diverging", "energy", "n_steps"}
    missing_sample_stats = sorted(required_sample_stats - available_sample_stats)
    step_size_stats = sorted(
        available_sample_stats & {"step_size", "step_size_bar", "step_size_nom"}
    )
    sample_stats_complete = bool(not missing_sample_stats and step_size_stats)
    divergence_count = None
    bfmi = np.array([np.nan])
    maxdepth_fraction_by_chain = []
    maxdepth_detection_method = None
    sample_stats_finite = False
    if sample_stats_complete:
        stats_to_check = required_sample_stats | set(step_size_stats)
        arrays_to_check = {
            name: np.asarray(idata.sample_stats[name], dtype=float)
            for name in stats_to_check
        }
        sample_stats_finite = bool(
            all(np.isfinite(values).all() for values in arrays_to_check.values())
            and (arrays_to_check["n_steps"] > 0).all()
            and all((arrays_to_check[name] > 0).all() for name in step_size_stats)
        )
        divergence_values_valid = bool(
            np.isin(arrays_to_check["diverging"], [0.0, 1.0]).all()
        )
        sample_stats_finite = bool(sample_stats_finite and divergence_values_valid)
        divergence_count = int(arrays_to_check["diverging"].sum())
        try:
            bfmi = np.asarray(az.bfmi(idata), dtype=float)
        except Exception:
            bfmi = np.array([np.nan])
        # nutpie 0.16.11 exposes `maxdepth_reached` and `depth`. Prefer its
        # direct indicator; retain explicit cross-backend fallbacks, all
        # validated and recorded rather than silently passing a missing gate.
        if "maxdepth_reached" in available_sample_stats:
            reached = np.asarray(
                idata.sample_stats["maxdepth_reached"], dtype=float
            )
            sample_stats_finite = bool(
                sample_stats_finite and np.isfinite(reached).all()
                and np.isin(reached, [0.0, 1.0]).all()
            )
            maxdepth_fraction_by_chain = reached.mean(axis=1).astype(float).tolist()
            maxdepth_detection_method = "maxdepth_reached"
        elif "reached_max_treedepth" in available_sample_stats:
            reached = np.asarray(
                idata.sample_stats["reached_max_treedepth"], dtype=float
            )
            sample_stats_finite = bool(
                sample_stats_finite and np.isfinite(reached).all()
                and np.isin(reached, [0.0, 1.0]).all()
            )
            maxdepth_fraction_by_chain = reached.mean(axis=1).astype(float).tolist()
            maxdepth_detection_method = "reached_max_treedepth"
        elif "depth" in available_sample_stats:
            depth = np.asarray(idata.sample_stats["depth"], dtype=float)
            sample_stats_finite = bool(
                sample_stats_finite and np.isfinite(depth).all() and (depth >= 0).all()
            )
            maxdepth_fraction_by_chain = (
                depth >= MAX_TREE_DEPTH
            ).mean(axis=1).astype(float).tolist()
            maxdepth_detection_method = "depth"
        elif "tree_depth" in available_sample_stats:
            tree_depth = np.asarray(idata.sample_stats["tree_depth"], dtype=float)
            sample_stats_finite = bool(
                sample_stats_finite and np.isfinite(tree_depth).all()
                and (tree_depth >= 0).all()
            )
            maxdepth_fraction_by_chain = (
                tree_depth >= MAX_TREE_DEPTH
            ).mean(axis=1).astype(float).tolist()
            maxdepth_detection_method = "tree_depth"
        else:
            max_steps = 2 ** MAX_TREE_DEPTH - 1
            maxdepth_fraction_by_chain = (
                arrays_to_check["n_steps"] >= max_steps
            ).mean(axis=1).astype(float).tolist()
            maxdepth_detection_method = "n_steps_fallback"

    expected_shape = PROFILES[PROFILE]
    trace_shape_pass = bool(
        idata.posterior.sizes.get("chain") == expected_shape["chains"]
        and idata.posterior.sizes.get("draw") == expected_shape["draws"]
    )
    sampler_geometry_pass = bool(
        sample_stats_complete and sample_stats_finite and divergence_count == 0
        and bfmi.size == expected_shape["chains"]
        and np.isfinite(bfmi).all() and np.min(bfmi) >= BFMI_MIN
        and len(maxdepth_fraction_by_chain) == expected_shape["chains"]
        and max(maxdepth_fraction_by_chain) < MAX_TREE_DEPTH_FRACTION
        and trace_shape_pass
    )

    SMOKE_SMALL_CORE_FAMILIES = {
        "celltype_bmi_slope", "feature_sigma_bmi", "celltype_covariate_slope",
        "feature_sigma_covariate", "patient_global_sigma", "log_sigma_location",
        "log_sigma_celltype_location_sd", "log_sigma_celltype_raw",
        "log_sigma_celltype_location", "log_sigma_feature_scale",
        "nu_minus_two", "nu",
    }
    if BMI_SHAPE == "linear_quadratic":
        SMOKE_SMALL_CORE_FAMILIES |= {"celltype_bmi_quadratic", "feature_sigma_quadratic"}
    if INCLUDE_THETA:
        SMOKE_SMALL_CORE_FAMILIES |= {"celltype_theta_slope", "feature_sigma_theta"}

    smoke_global_pass = bool(
        diagnostic_values_finite
        and full_diagnostics["r_hat"].max() < SMOKE_GLOBAL_RHAT_HARD_MAX
        and full_diagnostics["ess_bulk"].min() >= SMOKE_GLOBAL_ESS_HARD_MIN
        and full_diagnostics["ess_tail"].min() >= SMOKE_GLOBAL_ESS_HARD_MIN
    )
    small = full_diagnostics[
        full_diagnostics["parameter_family"].isin(SMOKE_SMALL_CORE_FAMILIES)
    ]
    missing_small = sorted(SMOKE_SMALL_CORE_FAMILIES - set(small["parameter_family"]))
    smoke_small_pass = bool(
        not missing_small and (small["r_hat"] < SMOKE_SMALL_RHAT_MAX).all()
        and (small["ess_bulk"] >= SMOKE_SMALL_ESS_MIN).all()
        and (small["ess_tail"] >= SMOKE_SMALL_ESS_MIN).all()
    )
    estimand = full_diagnostics[
        full_diagnostics["parameter_family"].eq("feature_bmi_slope")
    ].copy()
    estimand_bad = (
        estimand["r_hat"].ge(SMOKE_ESTIMAND_RHAT_MAX)
        | estimand["ess_bulk"].lt(SMOKE_ESTIMAND_ESS_MIN)
        | estimand["ess_tail"].lt(SMOKE_ESTIMAND_ESS_MIN)
    )
    smoke_estimand_pass = bool(
        len(estimand) == len(model_arrays["features"])
        and float(estimand_bad.mean()) <= SMOKE_ESTIMAND_BAD_FRACTION_MAX
        and estimand["r_hat"].max() < SMOKE_ESTIMAND_RHAT_HARD_MAX
        and estimand["ess_bulk"].min() >= SMOKE_ESTIMAND_ESS_HARD_MIN
        and estimand["ess_tail"].min() >= SMOKE_ESTIMAND_ESS_HARD_MIN
    )
    vector = full_diagnostics[
        ~full_diagnostics["parameter_family"].isin(
            SMOKE_SMALL_CORE_FAMILIES | {"feature_bmi_slope"}
        )
    ].copy()
    vector["smoke_bad"] = (
        vector["r_hat"].ge(SMOKE_VECTOR_RHAT_MAX)
        | vector["ess_bulk"].lt(SMOKE_VECTOR_ESS_MIN)
        | vector["ess_tail"].lt(SMOKE_VECTOR_ESS_MIN)
    )
    vector_family_summary = vector.groupby("parameter_family", as_index=False).agg(
        n_rows=("parameter", "size"), maximum_rhat=("r_hat", "max"),
        minimum_bulk_ess=("ess_bulk", "min"), minimum_tail_ess=("ess_tail", "min"),
        fraction_smoke_bad=("smoke_bad", "mean"),
    )
    smoke_vector_pass = bool(
        len(vector_family_summary) > 0
        and vector_family_summary["fraction_smoke_bad"].le(SMOKE_VECTOR_BAD_FRACTION_MAX).all()
    )
    smoke_gate_pass = bool(
        PROFILE == "smoke" and not missing_derived and sampler_geometry_pass
        and smoke_global_pass and smoke_small_pass and smoke_estimand_pass
        and smoke_vector_pass
    )
    strict_all_parameter_pass = bool(
        diagnostic_values_finite and not missing_derived
        and (full_diagnostics["r_hat"] < FINAL_RHAT_MAX).all()
        and (full_diagnostics["ess_bulk"] >= FINAL_ESS_MIN).all()
        and (full_diagnostics["ess_tail"] >= FINAL_ESS_MIN).all()
    )
    standard_gate_pass = bool(
        PROFILE == "standard" and sampler_geometry_pass and strict_all_parameter_pass
    )
    publication_convergence_pass = bool(
        PROFILE in {"publication", "publication_long"}
        and sampler_geometry_pass and strict_all_parameter_pass
    )
    current_profile_gate_pass = {
        "smoke": smoke_gate_pass,
        "standard": standard_gate_pass,
        "publication": publication_convergence_pass,
        "publication_long": publication_convergence_pass,
    }[PROFILE]

    family_summary = full_diagnostics.groupby("parameter_family", as_index=False).agg(
        n_rows=("parameter", "size"), maximum_rhat=("r_hat", "max"),
        minimum_bulk_ess=("ess_bulk", "min"), minimum_tail_ess=("ess_tail", "min"),
    ).sort_values(["maximum_rhat", "minimum_bulk_ess"], ascending=[False, True])
    atomic_csv(family_summary, OUTPUT_DIR / "diagnostic_family_summary.csv")
    atomic_csv(vector_family_summary, OUTPUT_DIR / "smoke_vector_family_summary.csv")
    gate_table = pd.DataFrame([
        {"gate": "sampler_geometry", "pass": sampler_geometry_pass},
        {"gate": "smoke_global_hard_bounds", "pass": smoke_global_pass},
        {"gate": "smoke_small_core", "pass": smoke_small_pass},
        {"gate": "smoke_feature_estimand", "pass": smoke_estimand_pass},
        {"gate": "smoke_vector_families", "pass": smoke_vector_pass},
        {"gate": "strict_all_parameters", "pass": strict_all_parameter_pass},
        {"gate": "current_profile", "pass": current_profile_gate_pass},
    ])
    atomic_csv(gate_table, OUTPUT_DIR / "diagnostic_gate_table.csv")
    diagnostics_record = {
        "analysis_version": ANALYSIS_VERSION,
        "model_architecture": MODEL_ARCHITECTURE,
        "model_fingerprint": MODEL_FINGERPRINT,
        "analysis_lock_sha256": ANALYSIS_LOCK_SHA256,
        "analysis_input_sha256": preflight_manifest["analysis_input_sha256"],
        "run_spec": RUN_SPEC, "role": ACTIVE_SPEC["role"],
        "analysis_family": ACTIVE_SPEC["analysis_family"], "profile": PROFILE,
        "posterior_trace_sha256": sha256_file(TRACE_PATH),
        "notebook_release_status": NOTEBOOK_RELEASE.get("status"),
        "notebook_release_sha256": NOTEBOOK_RELEASE.get("sha256"),
        "available_sample_stats": sorted(available_sample_stats),
        "missing_required_sample_stats": missing_sample_stats,
        "available_step_size_stats": step_size_stats,
        "sample_stats_complete": sample_stats_complete,
        "sample_stats_finite": sample_stats_finite,
        "expected_chains": expected_shape["chains"],
        "observed_chains": idata.posterior.sizes.get("chain"),
        "expected_draws_per_chain": expected_shape["draws"],
        "observed_draws_per_chain": idata.posterior.sizes.get("draw"),
        "trace_shape_pass": trace_shape_pass,
        "divergences": divergence_count,
        "bfmi_by_chain": bfmi.tolist(),
        "min_bfmi": float(np.nanmin(bfmi)),
        "maxdepth_fraction_by_chain": maxdepth_fraction_by_chain,
        "maxdepth_detection_method": maxdepth_detection_method,
        "missing_required_derived_diagnostics": missing_derived,
        "max_all_parameter_rhat": float(full_diagnostics["r_hat"].max()),
        "min_all_parameter_ess_bulk": float(full_diagnostics["ess_bulk"].min()),
        "min_all_parameter_ess_tail": float(full_diagnostics["ess_tail"].min()),
        "smoke_gate_pass": smoke_gate_pass,
        "standard_gate_pass": standard_gate_pass,
        "publication_convergence_pass": publication_convergence_pass,
        "current_profile_gate_pass": current_profile_gate_pass,
        "selection_status": "withheld_not_computed",
        "selection_revealed": False,
        "selection_reveal_utc": None,
        "promotion_evidence": promotion_evidence,
    }
    atomic_json(diagnostics_record, OUTPUT_DIR / "run_diagnostics.json")
    print(json.dumps(diagnostics_record, indent=2))


In [ ]:
# 6a. Optional failed-fit triage (diagnostic metrics only; no effects)
if RUN_TRIAGE:
    if idata is None:
        raise ValueError("RUN_TRIAGE requires a loaded trace")
    if diagnostics_record["current_profile_gate_pass"]:
        print("The current gate passed; triage is unnecessary.")
    else:
        worst = full_diagnostics.sort_values(
            ["r_hat", "ess_bulk"], ascending=[False, True]
        ).head(100)
        chain_rows = []
        for variable in diagnostic_names:
            if variable not in idata.posterior:
                continue
            values = idata.posterior[variable]
            parameter_dims = [d for d in values.dims if d not in {"chain", "draw"}]
            array = np.asarray(values.transpose("chain", "draw", *parameter_dims), float)
            array = array.reshape(array.shape[0], array.shape[1], -1)
            chain_means = array.mean(axis=1)
            pooled_sd = array.reshape(-1, array.shape[-1]).std(axis=0, ddof=1)
            separation = (chain_means.max(axis=0) - chain_means.min(axis=0)) / np.where(
                pooled_sd > 0, pooled_sd, np.nan
            )
            chain_rows.append({
                "variable": variable,
                "maximum_chain_mean_separation_in_posterior_sd": float(np.nanmax(separation)),
            })
        chain_separation = pd.DataFrame(chain_rows).sort_values(
            "maximum_chain_mean_separation_in_posterior_sd", ascending=False
        )
        geometry_rows = []
        for variable in sorted(available_sample_stats):
            values = np.asarray(idata.sample_stats[variable])
            if not np.issubdtype(values.dtype, np.number):
                continue
            for chain in range(values.shape[0]):
                geometry_rows.append({
                    "variable": variable, "chain": chain,
                    "median": float(np.nanmedian(values[chain])),
                    "maximum": float(np.nanmax(values[chain])),
                })
        atomic_csv(worst, OUTPUT_DIR / "triage_worst_diagnostic_rows.csv")
        atomic_csv(chain_separation, OUTPUT_DIR / "triage_chain_separation.csv")
        atomic_csv(pd.DataFrame(geometry_rows), OUTPUT_DIR / "triage_sampler_geometry.csv")
        display(family_summary, worst, chain_separation)
else:
    print("Optional triage disabled.")


In [ ]:
# 7. Corrected residual-based posterior predictive checks (PPC v2)
# Uses residual dispersion, studentized tails, and raw/conditional dependence.

PPC_V2_VERSION = "residual-symmetric-v2"
PPC_V2_BATCH_MB = 256
PPC_V2_ALPHA = 0.025

def _ppc_v2_finite(name, values):
    values = np.asarray(values)
    if values.size == 0 or not np.issubdtype(values.dtype, np.number):
        raise RuntimeError(f"{name} is empty or nonnumeric")
    if not np.isfinite(values.astype(float, copy=False)).all():
        raise RuntimeError(f"{name} contains nonfinite values")
    return values

def _ppc_v2_atomic_npz(path: Path, **arrays):
    temporary = path.with_name(f".{path.stem}.tmp.npz")
    if temporary.exists():
        temporary.unlink()
    np.savez_compressed(temporary, **arrays)
    temporary.replace(path)

def _ppc_v2_posterior_mean(posterior, name, dimensions):
    array = posterior[name].mean(dim=("chain", "draw"))
    return np.asarray(array.transpose(*dimensions), dtype=float)

def _ppc_v2_flat_draws(posterior_batch, name, dimensions):
    array = posterior_batch[name].transpose("chain", "draw", *dimensions)
    values = np.asarray(array, dtype=float)
    return values.reshape((-1,) + values.shape[2:])

def _ppc_v2_residual_statistics(batch_feature_patient):
    values = _ppc_v2_finite("residual matrix", batch_feature_patient).astype(float)
    residual_sd = values.std(axis=2, ddof=1)
    if (residual_sd <= 0).any():
        raise RuntimeError("A residual feature has zero variance")
    centered = values - values.mean(axis=2, keepdims=True)
    studentized_max_abs = np.max(np.abs(centered), axis=2) / residual_sd
    return residual_sd, studentized_max_abs

def _ppc_v2_dependence_count(batch_feature_patient, columns, threshold):
    values = _ppc_v2_finite(
        "dependence matrix", batch_feature_patient[:, columns, :]
    ).astype(float)
    if values.shape[1] < 2:
        return np.zeros(values.shape[0], dtype=np.int32)
    values = values - values.mean(axis=2, keepdims=True)
    norms = np.sqrt(np.sum(values * values, axis=2))
    if (norms <= 0).any():
        raise RuntimeError("A dependence feature has zero variance")
    numerator = np.einsum("bip,bjp->bij", values, values, optimize=True)
    denominator = norms[:, :, None] * norms[:, None, :]
    correlation = numerator / denominator
    upper = np.triu_indices(values.shape[1], k=1)
    return np.sum(
        np.abs(correlation[:, upper[0], upper[1]]) >= threshold,
        axis=1,
    ).astype(np.int32)

def _ppc_v2_upper_p(reference, observed):
    reference = np.asarray(reference, dtype=float).reshape(-1)
    return float((1 + np.sum(reference >= observed)) / (len(reference) + 1))

if RUN_PPC:
    if idata is None:
        raise RuntimeError("Load the saved posterior trace before running PPC v2")

    diagnostics_path = OUTPUT_DIR / "run_diagnostics.json"
    diagnostics = json.loads(diagnostics_path.read_text(encoding="utf-8"))
    if diagnostics.get("current_profile_gate_pass") is not True:
        raise RuntimeError("PPC v2 requires a converged saved trace")
    if diagnostics.get("posterior_trace_sha256") != sha256_file(TRACE_PATH):
        raise RuntimeError("The loaded trace differs from run_diagnostics.json")

    requested_joint = int(
        PPC_JOINT_DRAWS_OVERRIDE
        if PPC_JOINT_DRAWS_OVERRIDE is not None
        else PPC_SETTINGS[PROFILE]["joint_draws"]
    )
    n_chains = int(idata.posterior.sizes["chain"])
    stored_draws = int(idata.posterior.sizes["draw"])
    if requested_joint % (2 * n_chains):
        raise ValueError("PPC draws must split evenly by chain and into two halves")
    per_chain = requested_joint // n_chains
    if per_chain > stored_draws:
        raise ValueError("The trace has too few posterior draws for this PPC")
    selected_positions = np.linspace(
        0, stored_draws - 1, per_chain
    ).round().astype(int)
    if len(np.unique(selected_positions)) != per_chain:
        raise RuntimeError("Posterior draw-position selection is not unique")

    data = model_arrays["data"]
    observed = data["score_model"].to_numpy(float)
    feature_idx = data["feature_idx"].to_numpy(int)
    patient_idx = data["patient_idx"].to_numpy(int)
    bmi_z = data["bmi_standardized"].to_numpy(float)
    covariates = data[["age_z", "sex_centered"]].to_numpy(float)
    n_observations = len(observed)
    n_features = len(model_arrays["features"])
    n_patients = len(model_arrays["patients"])
    n_celltypes = len(model_arrays["celltypes"])

    observation_grid = np.full((n_features, n_patients), -1, dtype=np.int64)
    pair_code = feature_idx * n_patients + patient_idx
    if np.unique(pair_code).size != n_observations:
        raise ValueError("Duplicate patient-feature observations")
    observation_grid[feature_idx, patient_idx] = np.arange(n_observations)
    if (observation_grid < 0).any():
        raise ValueError("PPC v2 requires a complete patient-feature rectangle")

    feature_columns_by_celltype = {
        j: np.flatnonzero(model_arrays["feature_to_celltype"] == j)
        for j in range(n_celltypes)
    }

    # Posterior-mean fitted value for the fixed observed residual matrix.
    posterior = idata.posterior
    baseline_hat = _ppc_v2_posterior_mean(
        posterior, "feature_baseline", ("feature",)
    )
    bmi_hat = _ppc_v2_posterior_mean(
        posterior, "feature_bmi_slope", ("feature",)
    )
    covariate_hat = _ppc_v2_posterior_mean(
        posterior, "feature_covariate_slope", ("covariate", "feature")
    )
    patient_hat = _ppc_v2_posterior_mean(
        posterior, "patient_global_effect", ("patient",)
    )
    mu_hat = (
        baseline_hat[feature_idx]
        + bmi_hat[feature_idx] * bmi_z
        + patient_hat[patient_idx]
        + np.einsum(
            "oc,co->o", covariates, covariate_hat[:, feature_idx], optimize=True
        )
    )
    if "feature_bmi_quadratic" in posterior:
        quadratic_hat = _ppc_v2_posterior_mean(
            posterior, "feature_bmi_quadratic", ("feature",)
        )
        mu_hat += (
            quadratic_hat[feature_idx]
            * data["bmi_quadratic_orthogonal"].to_numpy(float)
        )
    if "feature_theta_slope" in posterior:
        theta_hat = _ppc_v2_posterior_mean(
            posterior, "feature_theta_slope", ("feature",)
        )
        mu_hat += theta_hat[feature_idx] * data["theta_z"].to_numpy(float)

    observed_feature_matrix = observed[observation_grid]
    observed_residual_matrix = (observed - mu_hat)[observation_grid]
    observed_residual_sd, observed_studentized_tail = (
        _ppc_v2_residual_statistics(observed_residual_matrix[None, :, :])
    )
    observed_residual_sd = observed_residual_sd[0]
    observed_studentized_tail = observed_studentized_tail[0]

    observed_raw_dependence = np.array([
        _ppc_v2_dependence_count(
            observed_feature_matrix[None, :, :], columns,
            PPC_CORRELATION_THRESHOLD,
        )[0]
        for columns in feature_columns_by_celltype.values()
    ], dtype=np.int32)
    observed_residual_dependence = np.array([
        _ppc_v2_dependence_count(
            observed_residual_matrix[None, :, :], columns,
            PPC_CORRELATION_THRESHOLD,
        )[0]
        for columns in feature_columns_by_celltype.values()
    ], dtype=np.int32)

    replicate_residual_sd = np.empty(
        (requested_joint, n_features), dtype=np.float32
    )
    replicate_studentized_tail = np.empty(
        (requested_joint, n_features), dtype=np.float32
    )
    replicate_raw_dependence = np.empty(
        (requested_joint, n_celltypes), dtype=np.int32
    )
    replicate_residual_dependence = np.empty(
        (requested_joint, n_celltypes), dtype=np.int32
    )
    replicate_chain = np.empty(requested_joint, dtype=np.int16)
    replicate_draw_position = np.empty(requested_joint, dtype=np.int32)
    predictive_le_count = np.zeros(n_observations, dtype=np.int32)

    # Allow room for y_rep, mu, and feature-by-patient working arrays.
    bytes_per_position = max(1, 4 * 8 * n_observations * n_chains)
    positions_per_batch = max(
        1, int(PPC_V2_BATCH_MB * 1024**2 // bytes_per_position)
    )
    n_batches = math.ceil(per_chain / positions_per_batch)
    print(
        f"Corrected PPC v2: {requested_joint} joint datasets "
        f"in {n_batches} memory-bounded batches"
    )

    cursor = 0
    batch_records = []
    for batch_number, start in enumerate(
        range(0, per_chain, positions_per_batch)
    ):
        if batch_number == 0 or (batch_number + 1) % 5 == 0 or (batch_number + 1) == n_batches:
            print(f"PPC v2 batch {batch_number + 1}/{n_batches}")
        positions = selected_positions[start:start + positions_per_batch]
        posterior_batch_dataset = posterior.isel(draw=positions)
        posterior_batch = az.InferenceData(posterior=posterior_batch_dataset)
        seed = RUN_SEED + 70_000 + batch_number
        with model:
            prediction = pm.sample_posterior_predictive(
                posterior_batch,
                var_names=["y_obs"],
                random_seed=seed,
                return_inferencedata=False,
                progressbar=False,
            )

        replicated = np.asarray(prediction["y_obs"], dtype=float)
        expected_shape = (n_chains, len(positions), n_observations)
        if replicated.shape != expected_shape:
            raise RuntimeError(
                f"Unexpected PPC shape {replicated.shape}; expected {expected_shape}"
            )
        replicated = replicated.reshape(-1, n_observations)
        n_batch = len(replicated)
        stop = cursor + n_batch

        baseline_draw = _ppc_v2_flat_draws(
            posterior_batch_dataset, "feature_baseline", ("feature",)
        )
        bmi_draw = _ppc_v2_flat_draws(
            posterior_batch_dataset, "feature_bmi_slope", ("feature",)
        )
        covariate_draw = _ppc_v2_flat_draws(
            posterior_batch_dataset,
            "feature_covariate_slope",
            ("covariate", "feature"),
        )
        patient_draw = _ppc_v2_flat_draws(
            posterior_batch_dataset, "patient_global_effect", ("patient",)
        )
        mu_draw = (
            baseline_draw[:, feature_idx]
            + bmi_draw[:, feature_idx] * bmi_z[None, :]
            + patient_draw[:, patient_idx]
            + np.einsum(
                "oc,bco->bo",
                covariates,
                covariate_draw[:, :, feature_idx],
                optimize=True,
            )
        )
        if "feature_bmi_quadratic" in posterior_batch_dataset:
            quadratic_draw = _ppc_v2_flat_draws(
                posterior_batch_dataset,
                "feature_bmi_quadratic",
                ("feature",),
            )
            mu_draw += (
                quadratic_draw[:, feature_idx]
                * data["bmi_quadratic_orthogonal"].to_numpy(float)[None, :]
            )
        if "feature_theta_slope" in posterior_batch_dataset:
            theta_draw = _ppc_v2_flat_draws(
                posterior_batch_dataset, "feature_theta_slope", ("feature",)
            )
            mu_draw += (
                theta_draw[:, feature_idx]
                * data["theta_z"].to_numpy(float)[None, :]
            )

        predictive_le_count += np.sum(
            replicated <= observed[None, :], axis=0
        ).astype(np.int32)

        raw_feature_matrix = replicated[:, observation_grid]
        for j, columns in feature_columns_by_celltype.items():
            replicate_raw_dependence[cursor:stop, j] = (
                _ppc_v2_dependence_count(
                    raw_feature_matrix, columns, PPC_CORRELATION_THRESHOLD
                )
            )
        del raw_feature_matrix

        # Conditional residuals: y_rep - mu for the same posterior draw.
        replicated -= mu_draw
        residual_feature_matrix = replicated[:, observation_grid]
        batch_sd, batch_tail = _ppc_v2_residual_statistics(
            residual_feature_matrix
        )
        replicate_residual_sd[cursor:stop] = batch_sd
        replicate_studentized_tail[cursor:stop] = batch_tail
        for j, columns in feature_columns_by_celltype.items():
            replicate_residual_dependence[cursor:stop, j] = (
                _ppc_v2_dependence_count(
                    residual_feature_matrix,
                    columns,
                    PPC_CORRELATION_THRESHOLD,
                )
            )

        replicate_chain[cursor:stop] = np.repeat(
            np.arange(n_chains), len(positions)
        )
        replicate_draw_position[cursor:stop] = np.tile(
            positions, n_chains
        )
        batch_records.append({
            "batch": batch_number,
            "seed": seed,
            "positions_per_chain": len(positions),
            "joint_datasets": n_batch,
        })
        cursor = stop

        del (
            posterior_batch, posterior_batch_dataset, prediction, replicated,
            residual_feature_matrix, mu_draw, baseline_draw, bmi_draw,
            covariate_draw, patient_draw, batch_sd, batch_tail,
        )
        gc.collect()

    if cursor != requested_joint:
        raise RuntimeError(f"Generated {cursor}; expected {requested_joint}")

    calibration_mask = np.zeros(requested_joint, dtype=bool)
    evaluation_mask = np.zeros(requested_joint, dtype=bool)
    for chain in range(n_chains):
        indices = np.flatnonzero(replicate_chain == chain)
        calibration_mask[indices[::2]] = True
        evaluation_mask[indices[1::2]] = True
    if (
        calibration_mask.sum() != requested_joint // 2
        or evaluation_mask.sum() != requested_joint // 2
        or np.any(calibration_mask & evaluation_mask)
    ):
        raise RuntimeError("The balanced PPC calibration/evaluation split failed")

    inventory = feature_inventory.copy().set_index("global_feature").loc[
        model_arrays["features"]
    ].reset_index()
    feature_checks = inventory.copy()
    feature_checks["observed_residual_sd"] = observed_residual_sd
    feature_checks["replicated_residual_sd_median"] = np.median(
        replicate_residual_sd, axis=0
    )
    feature_checks["observed_studentized_max_abs"] = observed_studentized_tail
    feature_checks["replicated_studentized_max_abs_median"] = np.median(
        replicate_studentized_tail, axis=0
    )

    domain_rows = []

    def add_count_rows(domain, observed_celltype, replicated_celltype):
        labels = ["__GLOBAL__", *model_arrays["celltypes"]]
        observed_all = np.r_[observed_celltype.sum(), observed_celltype]
        replicated_all = np.column_stack([
            replicated_celltype.sum(axis=1), replicated_celltype
        ])
        for j, label in enumerate(labels):
            reference = replicated_all[:, j]
            probability = _ppc_v2_upper_p(reference, observed_all[j])
            mcse = float(
                np.sqrt(probability * (1 - probability) / len(reference))
            )
            domain_rows.append({
                "domain": domain,
                "global_celltype": label,
                "observed_count": int(observed_all[j]),
                "replicated_q025": float(np.quantile(reference, 0.025)),
                "replicated_median": float(np.median(reference)),
                "replicated_q975": float(np.quantile(reference, 0.975)),
                "upper_tail_p": probability,
                "mcse_approx": mcse,
                "review_flag_p_le_0_025": bool(probability <= PPC_V2_ALPHA),
            })

    def calibrated_feature_domain(domain, observed_statistic, replicated_statistic):
        calibration = replicated_statistic[calibration_mask]
        evaluation = replicated_statistic[evaluation_mask]
        low, high = np.quantile(calibration, [0.025, 0.975], axis=0)
        observed_flags = (
            (observed_statistic < low) | (observed_statistic > high)
        )
        evaluation_flags = (
            (evaluation < low[None, :]) | (evaluation > high[None, :])
        )
        observed_celltype = np.array([
            observed_flags[feature_columns_by_celltype[j]].sum()
            for j in range(n_celltypes)
        ], dtype=int)
        evaluation_celltype = np.column_stack([
            evaluation_flags[:, feature_columns_by_celltype[j]].sum(axis=1)
            for j in range(n_celltypes)
        ])
        add_count_rows(domain, observed_celltype, evaluation_celltype)
        return low, high, observed_flags

    dispersion_low, dispersion_high, dispersion_flags = (
        calibrated_feature_domain(
            "residual_dispersion",
            observed_residual_sd,
            replicate_residual_sd,
        )
    )
    tail_low, tail_high, tail_flags = calibrated_feature_domain(
        "studentized_residual_tails",
        observed_studentized_tail,
        replicate_studentized_tail,
    )
    feature_checks["residual_sd_calibration_q025"] = dispersion_low
    feature_checks["residual_sd_calibration_q975"] = dispersion_high
    feature_checks["residual_dispersion_outside"] = dispersion_flags
    feature_checks["studentized_tail_calibration_q025"] = tail_low
    feature_checks["studentized_tail_calibration_q975"] = tail_high
    feature_checks["studentized_tail_outside"] = tail_flags

    add_count_rows(
        "raw_outcome_dependence",
        observed_raw_dependence,
        replicate_raw_dependence[evaluation_mask],
    )
    add_count_rows(
        "conditional_residual_dependence",
        observed_residual_dependence,
        replicate_residual_dependence[evaluation_mask],
    )

    predictive_cdf = (1 + predictive_le_count) / (requested_joint + 1)
    covered_90 = (predictive_cdf >= 0.05) & (predictive_cdf <= 0.95)
    feature_checks["predictive_90pct_coverage"] = np.array([
        covered_90[observation_grid[j]].mean() for j in range(n_features)
    ])

    domain_checks = pd.DataFrame(domain_rows)
    global_checks = domain_checks[
        domain_checks["global_celltype"].eq("__GLOBAL__")
    ].copy()
    review_domains = global_checks.loc[
        global_checks["review_flag_p_le_0_025"], "domain"
    ].tolist()
    global_tail_row = global_checks.loc[
        global_checks["domain"].eq("studentized_residual_tails")
    ]
    if len(global_tail_row) != 1:
        raise RuntimeError("Expected exactly one global tail-check row")
    tail_adequacy_pass = bool(
        not global_tail_row.iloc[0]["review_flag_p_le_0_025"]
    )

    feature_path = OUTPUT_DIR / "ppc_v2_feature_checks.csv"
    domain_path = OUTPUT_DIR / "ppc_v2_domain_checks.csv"
    discrepancy_path = OUTPUT_DIR / "ppc_v2_joint_discrepancies.npz"
    summary_path = OUTPUT_DIR / "ppc_v2_summary.json"
    atomic_csv(feature_checks, feature_path)
    atomic_csv(domain_checks, domain_path)
    _ppc_v2_atomic_npz(
        discrepancy_path,
        residual_sd=replicate_residual_sd,
        studentized_max_abs=replicate_studentized_tail,
        raw_dependence_count=replicate_raw_dependence,
        residual_dependence_count=replicate_residual_dependence,
        chain=replicate_chain,
        draw_position=replicate_draw_position,
        calibration_mask=calibration_mask,
        evaluation_mask=evaluation_mask,
        observed_residual_sd=observed_residual_sd,
        observed_studentized_max_abs=observed_studentized_tail,
        observed_raw_dependence=observed_raw_dependence,
        observed_residual_dependence=observed_residual_dependence,
    )

    summary = {
        "ppc_version": PPC_V2_VERSION,
        "analysis_version": ANALYSIS_VERSION,
        "model_architecture": MODEL_ARCHITECTURE,
        "model_fingerprint": MODEL_FINGERPRINT,
        "analysis_lock_sha256": ANALYSIS_LOCK_SHA256,
        "run_spec": RUN_SPEC,
        "profile": PROFILE,
        "posterior_trace_sha256": sha256_file(TRACE_PATH),
        "diagnostics_sha256": sha256_file(diagnostics_path),
        "nu_scope": "celltype_within_separate_compartment",
        "joint_draws": requested_joint,
        "calibration_draws": int(calibration_mask.sum()),
        "evaluation_draws": int(evaluation_mask.sum()),
        "n_features": n_features,
        "n_celltypes": n_celltypes,
        "correlation_threshold": PPC_CORRELATION_THRESHOLD,
        "observed_raw_high_correlation_pairs": int(
            observed_raw_dependence.sum()
        ),
        "observed_residual_high_correlation_pairs": int(
            observed_residual_dependence.sum()
        ),
        "median_predictive_90pct_coverage": float(
            feature_checks["predictive_90pct_coverage"].median()
        ),
        "minimum_predictive_90pct_coverage": float(
            feature_checks["predictive_90pct_coverage"].min()
        ),
        "global_domains_flagged_for_review": review_domains,
        "tail_adequacy_pass": tail_adequacy_pass,
        "decision": (
            "CELLTYPE_NU_TAIL_CHECK_PASSED"
            if tail_adequacy_pass
            else "CELLTYPE_NU_TAIL_MISMATCH_REMAINS"
        ),
        "important": (
            "This v2.2 notebook is a pre-result model-adequacy comparison. "
            "The corrected PPC does not overwrite v2.1c artifacts and does "
            "not authorize BMI-effect reveal by itself."
        ),
        "feature_checks_sha256": sha256_file(feature_path),
        "domain_checks_sha256": sha256_file(domain_path),
        "joint_discrepancies_sha256": sha256_file(discrepancy_path),
        "batches": batch_records,
    }
    atomic_json(summary, summary_path)

    display(domain_checks)
    print(json.dumps(summary, indent=2))
    print("Saved corrected PPC v2 outputs in:", OUTPUT_DIR)
else:
    print("Corrected PPC v2 disabled; set WORKFLOW_STAGE='ppc'.")


In [ ]:
# 7a. Review corrected PPC v2 and cell-type tail parameters
if RUN_PPC_REVIEW:
    summary_path = OUTPUT_DIR / "ppc_v2_summary.json"
    domain_path = OUTPUT_DIR / "ppc_v2_domain_checks.csv"
    feature_path = OUTPUT_DIR / "ppc_v2_feature_checks.csv"
    for path in (summary_path, domain_path, feature_path):
        if not path.exists():
            raise FileNotFoundError(
                f"Missing {path.name}; run WORKFLOW_STAGE='ppc' first"
            )
    saved_ppc = json.loads(summary_path.read_text(encoding="utf-8"))
    if saved_ppc.get("posterior_trace_sha256") != sha256_file(TRACE_PATH):
        raise RuntimeError("Saved PPC v2 does not belong to this trace")
    if saved_ppc.get("model_fingerprint") != MODEL_FINGERPRINT:
        raise RuntimeError("Saved PPC v2 belongs to another model")
    domain_checks = pd.read_csv(domain_path)
    feature_checks = pd.read_csv(feature_path)
    display(Markdown(
        f"### Cell-type-ν tail decision: `{saved_ppc['decision']}`\n\n"
        f"- profile: `{saved_ppc['profile']}`\n"
        f"- joint replicated datasets: `{saved_ppc['joint_draws']}`\n"
        f"- global tail adequacy: `{saved_ppc['tail_adequacy_pass']}`\n"
        f"- domains still requiring review: "
        f"`{saved_ppc['global_domains_flagged_for_review']}`"
    ))
    print("CORRECTED PPC V2 BY DOMAIN AND CELL TYPE")
    display(domain_checks.sort_values(
        ["domain", "upper_tail_p", "global_celltype"]
    ))
    print("CELL-TYPE STUDENT-t DEGREES OF FREEDOM")
    display(az.summary(
        idata, var_names=["nu"], kind="all", round_to=4
    ))
    identity = [column for column in (
        "global_celltype", "global_feature", "CellType", "signature"
    ) if column in feature_checks.columns]
    print("FEATURES WITH THE LARGEST OBSERVED STUDENTIZED TAILS")
    display(feature_checks.sort_values(
        "observed_studentized_max_abs", ascending=False
    )[identity + [
        "observed_studentized_max_abs",
        "studentized_tail_calibration_q975",
        "studentized_tail_outside",
        "predictive_90pct_coverage",
    ]].head(30))
else:
    print("PPC v2 review disabled; set WORKFLOW_STAGE='ppc_review'.")


In [ ]:
import json
import numpy as np
import pandas as pd
import xarray as xr

ALPHA = 0.025
BATCH_SIZE = 32

ppc_npz_path = OUTPUT_DIR / "ppc_v2_joint_discrepancies.npz"
ppc_summary_path = OUTPUT_DIR / "ppc_v2_summary.json"

if not ppc_npz_path.exists():
    raise FileNotFoundError("Run the existing PPC once before this audit.")

saved_meta = json.loads(ppc_summary_path.read_text())
if saved_meta["model_fingerprint"] != MODEL_FINGERPRINT:
    raise RuntimeError("Saved PPC belongs to a different model.")

if saved_meta["posterior_trace_sha256"] != sha256_file(TRACE_PATH):
    raise RuntimeError("Saved PPC belongs to a different trace.")

with np.load(ppc_npz_path, allow_pickle=False) as z:
    rep_sd = z["residual_sd"].copy()
    rep_tail = z["studentized_max_abs"].copy()
    rep_raw_dependence = z["raw_dependence_count"].copy()
    rep_residual_dependence = z["residual_dependence_count"].copy()
    replicate_chain = z["chain"].astype(int).copy()
    replicate_draw = z["draw_position"].astype(int).copy()
    calibration_mask = z["calibration_mask"].astype(bool).copy()
    evaluation_mask = z["evaluation_mask"].astype(bool).copy()
    observed_raw_dependence = z["observed_raw_dependence"].astype(int).copy()

evaluation_index = np.flatnonzero(evaluation_mask)
n_paired = len(evaluation_index)

data = model_arrays["data"]
posterior = idata.posterior

observed = data["score_model"].to_numpy(float)
feature_idx = data["feature_idx"].to_numpy(int)
patient_idx = data["patient_idx"].to_numpy(int)
bmi_z = data["bmi_standardized"].to_numpy(float)
covariates = data[["age_z", "sex_centered"]].to_numpy(float)

n_features = len(model_arrays["features"])
n_patients = len(model_arrays["patients"])
n_celltypes = len(model_arrays["celltypes"])

observation_grid = np.full(
    (n_features, n_patients), -1, dtype=np.int64
)
observation_grid[feature_idx, patient_idx] = np.arange(len(observed))

if (observation_grid < 0).any():
    raise RuntimeError("Patient-feature observation grid is incomplete.")

feature_columns_by_celltype = {
    j: np.flatnonzero(model_arrays["feature_to_celltype"] == j)
    for j in range(n_celltypes)
}

def residual_statistics(values):
    sd = values.std(axis=2, ddof=1)
    if np.any(sd <= 0):
        raise RuntimeError("Zero residual SD encountered.")
    centered = values - values.mean(axis=2, keepdims=True)
    tail = np.max(np.abs(centered), axis=2) / sd
    return sd, tail

def dependence_count(values, columns):
    x = values[:, columns, :]
    if x.shape[1] < 2:
        return np.zeros(x.shape[0], dtype=np.int32)

    x = x - x.mean(axis=2, keepdims=True)
    norms = np.sqrt(np.sum(x * x, axis=2))
    numerator = np.einsum("bip,bjp->bij", x, x, optimize=True)
    denominator = norms[:, :, None] * norms[:, None, :]
    correlation = numerator / denominator

    upper = np.triu_indices(x.shape[1], k=1)
    return np.sum(
        np.abs(correlation[:, upper[0], upper[1]])
        >= PPC_CORRELATION_THRESHOLD,
        axis=1,
    ).astype(np.int32)

observed_sd_draw = np.empty(
    (n_paired, n_features), dtype=np.float32
)
observed_tail_draw = np.empty(
    (n_paired, n_features), dtype=np.float32
)
observed_residual_dependence_draw = np.empty(
    (n_paired, n_celltypes), dtype=np.int32
)

for start in range(0, n_paired, BATCH_SIZE):
    stop = min(start + BATCH_SIZE, n_paired)
    saved_rows = evaluation_index[start:stop]

    paired_chain = xr.DataArray(
        replicate_chain[saved_rows], dims="ppc_pair"
    )
    paired_draw = xr.DataArray(
        replicate_draw[saved_rows], dims="ppc_pair"
    )
    draw_data = posterior.isel(
        chain=paired_chain,
        draw=paired_draw,
    )

    def get_draws(name, dimensions):
        return np.asarray(
            draw_data[name].transpose("ppc_pair", *dimensions),
            dtype=float,
        )

    baseline = get_draws("feature_baseline", ("feature",))
    bmi_slope = get_draws("feature_bmi_slope", ("feature",))
    covariate_slope = get_draws(
        "feature_covariate_slope",
        ("covariate", "feature"),
    )
    patient_effect = get_draws(
        "patient_global_effect", ("patient",)
    )

    mu_draw = (
        baseline[:, feature_idx]
        + bmi_slope[:, feature_idx] * bmi_z[None, :]
        + patient_effect[:, patient_idx]
        + np.einsum(
            "oc,bco->bo",
            covariates,
            covariate_slope[:, :, feature_idx],
            optimize=True,
        )
    )

    if "feature_bmi_quadratic" in posterior:
        quadratic = get_draws(
            "feature_bmi_quadratic", ("feature",)
        )
        mu_draw += (
            quadratic[:, feature_idx]
            * data["bmi_quadratic_orthogonal"]
            .to_numpy(float)[None, :]
        )

    if "feature_theta_slope" in posterior:
        theta_slope = get_draws(
            "feature_theta_slope", ("feature",)
        )
        mu_draw += (
            theta_slope[:, feature_idx]
            * data["theta_z"].to_numpy(float)[None, :]
        )

    # Crucial correction: y_obs - mu_draw for the same posterior draw
    observed_residual = observed[None, :] - mu_draw
    observed_matrix = observed_residual[:, observation_grid]

    batch_sd, batch_tail = residual_statistics(observed_matrix)
    observed_sd_draw[start:stop] = batch_sd
    observed_tail_draw[start:stop] = batch_tail

    for j, columns in feature_columns_by_celltype.items():
        observed_residual_dependence_draw[start:stop, j] = (
            dependence_count(observed_matrix, columns)
        )

    if start == 0 or stop == n_paired or stop % 320 == 0:
        print(f"Paired PPC: {stop}/{n_paired} draws")

def celltype_flag_counts(flags):
    return np.column_stack([
        flags[:, feature_columns_by_celltype[j]].sum(axis=1)
        for j in range(n_celltypes)
    ])

def calibrated_counts(observed_stat, replicated_stat):
    calibration = replicated_stat[calibration_mask]
    replicated_evaluation = replicated_stat[evaluation_mask]

    low, high = np.quantile(
        calibration, [0.025, 0.975], axis=0
    )

    observed_flags = (
        (observed_stat < low[None, :])
        | (observed_stat > high[None, :])
    )
    replicated_flags = (
        (replicated_evaluation < low[None, :])
        | (replicated_evaluation > high[None, :])
    )

    return (
        celltype_flag_counts(observed_flags),
        celltype_flag_counts(replicated_flags),
    )

observed_dispersion_count, replicated_dispersion_count = (
    calibrated_counts(observed_sd_draw, rep_sd)
)
observed_tail_count, replicated_tail_count = calibrated_counts(
    observed_tail_draw, rep_tail
)

replicated_raw_count = rep_raw_dependence[evaluation_mask]
observed_raw_count = np.broadcast_to(
    observed_raw_dependence,
    replicated_raw_count.shape,
)

replicated_residual_count = (
    rep_residual_dependence[evaluation_mask]
)
observed_residual_count = observed_residual_dependence_draw

rows = []
labels = ["__GLOBAL__", *model_arrays["celltypes"]]

def add_domain(domain, observed_celltype, replicated_celltype):
    observed_all = np.column_stack([
        observed_celltype.sum(axis=1),
        observed_celltype,
    ])
    replicated_all = np.column_stack([
        replicated_celltype.sum(axis=1),
        replicated_celltype,
    ])

    for j, label in enumerate(labels):
        obs = observed_all[:, j]
        rep = replicated_all[:, j]

        upper_p = (1 + np.sum(rep >= obs)) / (len(rep) + 1)
        lower_p = (1 + np.sum(rep <= obs)) / (len(rep) + 1)
        two_sided_p = min(1.0, 2 * min(upper_p, lower_p))

        rows.append({
            "domain": domain,
            "global_celltype": label,
            "observed_median": float(np.median(obs)),
            "observed_q025": float(np.quantile(obs, 0.025)),
            "observed_q975": float(np.quantile(obs, 0.975)),
            "replicated_median": float(np.median(rep)),
            "replicated_q025": float(np.quantile(rep, 0.025)),
            "replicated_q975": float(np.quantile(rep, 0.975)),
            "paired_upper_p": float(upper_p),
            "paired_lower_p": float(lower_p),
            "paired_two_sided_p": float(two_sided_p),
        })

add_domain(
    "residual_dispersion",
    observed_dispersion_count,
    replicated_dispersion_count,
)
add_domain(
    "studentized_residual_tails",
    observed_tail_count,
    replicated_tail_count,
)
add_domain(
    "raw_outcome_dependence",
    observed_raw_count,
    replicated_raw_count,
)
add_domain(
    "conditional_residual_dependence",
    observed_residual_count,
    replicated_residual_count,
)

paired_checks = pd.DataFrame(rows)

global_tail = paired_checks[
    paired_checks["domain"].eq("studentized_residual_tails")
    & paired_checks["global_celltype"].eq("__GLOBAL__")
].iloc[0]

tail_pass = bool(global_tail["paired_upper_p"] > ALPHA)

paired_summary = {
    "ppc_version": "draw-matched-paired-v1",
    "run_spec": RUN_SPEC,
    "profile": PROFILE,
    "paired_draws": n_paired,
    "tail_upper_p": float(global_tail["paired_upper_p"]),
    "tail_two_sided_p": float(
        global_tail["paired_two_sided_p"]
    ),
    "tail_adequacy_pass": tail_pass,
    "decision": (
        "PAIRED_TAIL_CHECK_PASSED"
        if tail_pass
        else "PAIRED_TAIL_MISMATCH_REMAINS"
    ),
    "model_refitted": False,
}

paired_checks.to_csv(
    OUTPUT_DIR / "ppc_v2_paired_domain_checks.csv",
    index=False,
)
(
    OUTPUT_DIR / "ppc_v2_paired_summary.json"
).write_text(json.dumps(paired_summary, indent=2))

display(
    paired_checks.sort_values(
        ["domain", "paired_upper_p", "global_celltype"]
    )
)
print(json.dumps(paired_summary, indent=2))


In [ ]:
# 9. Gate and atomically reveal posterior effects and ROPE-BFDR results
def _az_da_stat(da, statistic: str, method: str):
    dataset = xr.Dataset({"metric": da.astype(float)})
    function = az.ess if statistic == "ess" else az.mcse
    return function(dataset, var_names=["metric"], method=method)["metric"]

def effective_sample_boundary_guard(mean, n_eff, alpha=MC_ALPHA):
    """Conservative boundary guard; not an exact Clopper-Pearson interval."""
    n_eff = max(1.0, float(n_eff))
    mean = float(np.clip(mean, 0, 1))
    x = mean * n_eff
    lower = 0.0 if x <= 0 else float(beta_dist.ppf(alpha, x, n_eff - x + 1))
    upper = 1.0 if x >= n_eff else float(beta_dist.ppf(1 - alpha, x + 1, n_eff - x))
    return lower, upper

def rope_indicator_metrics(trace, rope):
    beta = trace.posterior["feature_bmi_slope"].transpose("chain", "draw", "feature")
    indicator = (np.abs(beta) <= float(rope)).astype(np.float32).rename("rope_indicator")
    probability = indicator.mean(("chain", "draw")).to_numpy()
    ess_raw = _az_da_stat(indicator, "ess", "bulk").to_numpy()
    mcse_raw = _az_da_stat(indicator, "mcse", "mean").to_numpy()
    beta_bulk = _az_da_stat(beta, "ess", "bulk").to_numpy()
    beta_tail = _az_da_stat(beta, "ess", "tail").to_numpy()
    n_total = beta.sizes["chain"] * beta.sizes["draw"]
    fallback = np.minimum(beta_bulk, beta_tail)
    ess = np.where(np.isfinite(ess_raw) & (ess_raw > 0), ess_raw, fallback)
    ess = np.minimum(ess, n_total)
    z_value = norm.ppf(1 - MC_ALPHA)
    rows = []
    for j, name in enumerate(beta.coords["feature"].to_numpy()):
        lower, upper = effective_sample_boundary_guard(probability[j], ess[j])
        raw = float(mcse_raw[j]) if np.isfinite(mcse_raw[j]) else 0.0
        guarded = max(raw, (upper - probability[j]) / z_value, (probability[j] - lower) / z_value)
        rows.append({
            "global_feature": str(name),
            "local_null_probability": float(probability[j]),
            "indicator_ess": float(ess[j]),
            "indicator_mcse_raw": raw,
            "indicator_mcse_guarded": float(guarded),
            "local_null_mc_lower": lower,
            "local_null_mc_upper": upper,
        })
    return indicator, pd.DataFrame(rows)

def deterministic_bfdr(local_null, feature_ids, q=Q_LEVEL):
    values = np.asarray(local_null, float)
    identifiers = np.asarray(feature_ids, str)
    if not len(values) or not np.isfinite(values).all():
        raise ValueError("Invalid local-null vector")
    order = np.lexsort((identifiers, values))
    cumulative = np.cumsum(values[order]) / np.arange(1, len(values) + 1)
    accepted = np.flatnonzero(cumulative <= q)
    k = int(accepted[-1] + 1) if len(accepted) else 0
    selected = np.zeros(len(values), dtype=bool)
    selected[order[:k]] = True
    return selected, order, cumulative, k

def bounded_series_mc(series, fallback_ess, alpha=MC_ALPHA):
    mean = float(series.mean())
    n_total = series.sizes["chain"] * series.sizes["draw"]
    ess = float(np.asarray(_az_da_stat(series.rename("metric"), "ess", "bulk")))
    mcse = float(np.asarray(_az_da_stat(series.rename("metric"), "mcse", "mean")))
    if not np.isfinite(ess) or ess <= 0: ess = float(fallback_ess)
    ess = min(ess, n_total)
    if not np.isfinite(mcse): mcse = 0.0
    guard_low, guard_high = effective_sample_boundary_guard(mean, ess, alpha)
    z_value = norm.ppf(1 - alpha)
    lower = max(0.0, min(mean - z_value * mcse, guard_low))
    upper = min(1.0, max(mean + z_value * mcse, guard_high))
    guarded_mcse = max(mcse, (upper - mean) / z_value, (mean - lower) / z_value)
    return {
        "mean": mean, "ess": ess, "mcse_raw": mcse,
        "mcse_guarded": guarded_mcse, "mc_lower": lower, "mc_upper": upper,
    }

def selection_sets(mask, feature_ids, cluster_ids):
    return (
        frozenset(np.asarray(feature_ids, str)[mask]),
        frozenset(np.asarray(cluster_ids, str)[mask]),
    )

def assess_rope_bfdr(trace, feature_ids, cluster_ids, rope, q=Q_LEVEL):
    if trace.posterior.sizes["chain"] < 4:
        raise ValueError("Reveal requires at least four chains")
    indicator, metrics = rope_indicator_metrics(trace, rope)
    trace_ids = np.asarray(indicator.coords["feature"].to_numpy(), str)
    feature_ids = np.asarray(feature_ids, str)
    raw_cluster_ids = np.asarray(cluster_ids, dtype=object)
    if pd.isna(raw_cluster_ids).any():
        raise ValueError("Missing correlation-cluster ID")
    cluster_ids = raw_cluster_ids.astype(str)
    if not np.array_equal(trace_ids, feature_ids):
        raise ValueError("Feature order differs from posterior coordinates")
    if np.any(np.char.strip(cluster_ids) == "") or np.any(np.char.lower(cluster_ids) == "nan"):
        raise ValueError("Empty/invalid correlation-cluster ID")
    probability = metrics["local_null_probability"].to_numpy()
    mask, order, cumulative, k = deterministic_bfdr(probability, feature_ids, q)
    next_index = None if k == len(probability) else int(order[k])
    critical = mask | (probability <= 0.10)
    if next_index is not None: critical[next_index] = True
    critical_metrics = metrics.loc[critical]
    indicator_precision_pass = bool(
        (critical_metrics["indicator_ess"] >= ROPE_INDICATOR_ESS_MIN).all()
        and (critical_metrics["indicator_mcse_guarded"] <= ROPE_LOCAL_NULL_MCSE_MAX).all()
    )
    if k:
        selected_series = indicator.isel(feature=np.flatnonzero(mask)).mean("feature")
        selected_mc = bounded_series_mc(selected_series, critical_metrics["indicator_ess"].min())
        selected_bound_pass = bool(
            selected_mc["mc_upper"] <= q and selected_mc["mcse_guarded"] <= BFDR_MCSE_MAX
        )
    else:
        selected_mc = {"mean": 0.0, "ess": None, "mcse_raw": 0.0, "mcse_guarded": 0.0, "mc_lower": 0.0, "mc_upper": 0.0}
        selected_bound_pass = True
    if next_index is not None:
        prefix = order[:k + 1]
        next_series = indicator.isel(feature=prefix).mean("feature")
        next_mc = bounded_series_mc(next_series, metrics.iloc[prefix]["indicator_ess"].min())
        next_bound_pass = bool(
            next_mc["mc_lower"] > q and next_mc["mcse_guarded"] <= BFDR_MCSE_MAX
        )
    else:
        next_mc = None
        next_bound_pass = True
    full_features, full_clusters = selection_sets(mask, feature_ids, cluster_ids)
    loo_rows = []
    for omitted in range(indicator.sizes["chain"]):
        keep = [j for j in range(indicator.sizes["chain"]) if j != omitted]
        loo_probability = indicator.isel(chain=keep).mean(("chain", "draw")).to_numpy()
        loo_mask, _, _, _ = deterministic_bfdr(loo_probability, feature_ids, q)
        feature_set, cluster_set = selection_sets(loo_mask, feature_ids, cluster_ids)
        loo_rows.append({
            "omitted_chain_position": omitted,
            "feature_set_identical": feature_set == full_features,
            "cluster_set_identical": cluster_set == full_clusters,
        })
    loo = pd.DataFrame(loo_rows)
    loo_pass = bool(loo[["feature_set_identical", "cluster_set_identical"]].to_numpy().all())
    ranks = np.empty(len(probability), int); ranks[order] = np.arange(1, len(probability) + 1)
    cumulative_by_feature = np.full(len(probability), np.nan); cumulative_by_feature[order] = cumulative
    metrics["bfdr_rank"] = ranks
    metrics["cumulative_bfdr_at_rank"] = cumulative_by_feature
    metrics["bfdr_selected"] = mask
    return {
        "pass": bool(indicator_precision_pass and selected_bound_pass and next_bound_pass and loo_pass),
        "rope": float(rope), "n_selected": k,
        "indicator_precision_pass": indicator_precision_pass,
        "selected_mc": selected_mc, "selected_bound_pass": selected_bound_pass,
        "next_prefix_mc": next_mc, "next_bound_pass": next_bound_pass,
        "loo_pass": loo_pass, "feature_metrics": metrics, "loo": loo,
        "minimum_critical_indicator_ess": float(critical_metrics["indicator_ess"].min()),
        "maximum_critical_indicator_mcse": float(critical_metrics["indicator_mcse_guarded"].max()),
    }

if RUN_RESULT_REVEAL:
    if not REVEAL_RESULTS_APPROVED:
        raise RuntimeError("Set REVEAL_RESULTS_APPROVED=True only after reviewing convergence and PPC")
    if PROFILE not in {"publication", "publication_long"} or idata is None:
        raise ValueError("Reveal requires the loaded publication trace")
    diagnostics_path = OUTPUT_DIR / "run_diagnostics.json"
    ppc_path = OUTPUT_DIR / "ppc_adequacy.json"
    if not diagnostics_path.exists() or not ppc_path.exists():
        raise FileNotFoundError("Publication diagnostics and PPC records are required")
    diagnostics = json.loads(diagnostics_path.read_text(encoding="utf-8"))
    ppc = json.loads(ppc_path.read_text(encoding="utf-8"))
    reveal_checks = {
        "publication_convergence": diagnostics.get("publication_convergence_pass") is True,
        "ppc_adequacy": ppc.get("ppc_adequacy_pass") is True,
        "diagnostic_trace": diagnostics.get("posterior_trace_sha256") == sha256_file(TRACE_PATH),
        "ppc_trace": ppc.get("posterior_trace_sha256") == sha256_file(TRACE_PATH),
        "diagnostic_lock": diagnostics.get("analysis_lock_sha256") == ANALYSIS_LOCK_SHA256,
        "ppc_lock": ppc.get("analysis_lock_sha256") == ANALYSIS_LOCK_SHA256,
        "diagnostic_fingerprint": diagnostics.get("model_fingerprint") == MODEL_FINGERPRINT,
        "ppc_fingerprint": ppc.get("model_fingerprint") == MODEL_FINGERPRINT,
        "diagnostic_profile": diagnostics.get("profile") == PROFILE,
        "ppc_profile": ppc.get("profile") == PROFILE,
        "ppc_diagnostics": ppc.get("diagnostics_sha256") == sha256_file(diagnostics_path),
        "diagnostic_notebook_verified": diagnostics.get("notebook_release_status") == "verified",
        "ppc_notebook_verified": ppc.get("notebook_release_status") == "verified",
        "diagnostic_notebook_release": diagnostics.get("notebook_release_sha256") == NOTEBOOK_RELEASE.get("sha256"),
        "ppc_notebook_release": ppc.get("notebook_release_sha256") == NOTEBOOK_RELEASE.get("sha256"),
        "feature_inventory": preflight_manifest.get("feature_inventory_sha256") == sha256_file(
            OUTPUT_DIR / "feature_inventory.csv"
        ),
        "notebook_verified": NOTEBOOK_RELEASE.get("status") == "verified",
    }
    failed = [name for name, passed in reveal_checks.items() if not passed]
    if failed:
        raise RuntimeError(f"Result reveal blocked; failed checks: {failed}")

    inventory = feature_inventory.set_index("global_feature").loc[model_arrays["features"]].reset_index()
    feature_ids = np.asarray(model_arrays["features"], str)
    cluster_ids = inventory["global_cluster_id"].astype(str).to_numpy()
    rope_assessments = {
        rope: assess_rope_bfdr(idata, feature_ids, cluster_ids, rope)
        for rope in ROPE_GRID
    }
    precision_gate = {
        "analysis_version": ANALYSIS_VERSION,
        "model_architecture": MODEL_ARCHITECTURE,
        "model_fingerprint": MODEL_FINGERPRINT,
        "notebook_release_sha256": NOTEBOOK_RELEASE["sha256"],
        "reveal_version": REVEAL_VERSION,
        "run_spec": RUN_SPEC,
        "profile": PROFILE,
        "analysis_lock_sha256": ANALYSIS_LOCK_SHA256,
        "posterior_trace_sha256": sha256_file(TRACE_PATH),
        "diagnostics_sha256": sha256_file(diagnostics_path),
        "ppc_adequacy_sha256": sha256_file(ppc_path),
        "all_displayed_ropes_pass": bool(all(value["pass"] for value in rope_assessments.values())),
        "by_rope": {
            str(rope): {
                "pass": value["pass"],
                "indicator_precision_pass": value["indicator_precision_pass"],
                "selected_bound_pass": value["selected_bound_pass"],
                "next_bound_pass": value["next_bound_pass"],
                "loo_pass": value["loo_pass"],
                "minimum_critical_indicator_ess": value["minimum_critical_indicator_ess"],
                "maximum_critical_indicator_mcse": value["maximum_critical_indicator_mcse"],
            }
            for rope, value in rope_assessments.items()
        },
        "selection_status": "withheld_precision_failed" if not all(value["pass"] for value in rope_assessments.values()) else "precision_pass",
    }
    atomic_json(precision_gate, OUTPUT_DIR / "reveal_precision_gate.json")
    if not precision_gate["all_displayed_ropes_pass"]:
        if PROFILE == "publication":
            raise RuntimeError(
                "ROPE/BFDR Monte Carlo precision failed. Use the predeclared "
                "PROFILE='publication_long' fresh fit of the same locked architecture; "
                "no effects or selection counts were written."
            )
        raise RuntimeError(
            "The predeclared publication_long trace still lacks ROPE/BFDR precision. "
            "Stop: no further automatic refit is authorized and no effects or "
            "selection counts were written."
        )

    beta = idata.posterior["feature_bmi_slope"].transpose("chain", "draw", "feature")
    samples = beta.stack(sample=("chain", "draw")).transpose("feature", "sample").to_numpy()
    hdi_object = az.hdi(beta, hdi_prob=0.95)
    hdi = hdi_object["feature_bmi_slope"].to_numpy() if hasattr(hdi_object, "data_vars") else hdi_object.to_numpy()
    results = inventory.copy()
    results["posterior_mean"] = samples.mean(axis=1)
    results["posterior_sd"] = samples.std(axis=1, ddof=1)
    results["hdi_low"] = hdi[:, 0]
    results["hdi_high"] = hdi[:, 1]
    results["posterior_prob_positive"] = np.mean(samples > 0, axis=1)
    results["posterior_prob_negative"] = np.mean(samples < 0, axis=1)
    for rope, assessment in rope_assessments.items():
        label = str(rope).replace(".", "p")
        metrics = assessment["feature_metrics"].set_index("global_feature").loc[feature_ids]
        for column in (
            "local_null_probability", "indicator_ess", "indicator_mcse_guarded",
            "local_null_mc_lower", "local_null_mc_upper", "bfdr_rank",
            "cumulative_bfdr_at_rank", "bfdr_selected",
        ):
            results[f"{column}_rope_{label}"] = metrics[column].to_numpy()
        atomic_csv(assessment["loo"], OUTPUT_DIR / f"revealed_loo_chain_rope_{label}.csv")
    results["r_hat"] = az.rhat(idata, var_names=["feature_bmi_slope"])["feature_bmi_slope"].to_numpy()
    results["ess_bulk"] = az.ess(idata, var_names=["feature_bmi_slope"], method="bulk")["feature_bmi_slope"].to_numpy()
    results["ess_tail"] = az.ess(idata, var_names=["feature_bmi_slope"], method="tail")["feature_bmi_slope"].to_numpy()

    results_path = OUTPUT_DIR / "revealed_feature_bmi_results.csv"
    atomic_csv(results, results_path)
    celltype_summary = az.summary(
        idata,
        var_names=[
            "celltype_bmi_slope", "feature_sigma_bmi", "celltype_covariate_slope",
            "feature_sigma_covariate", "patient_global_sigma", "log_sigma_location",
            "log_sigma_celltype_location_sd", "log_sigma_celltype_location",
            "log_sigma_feature_scale", "nu",
        ],
        round_to="none",
    ).reset_index(names="parameter")
    celltype_summary_path = OUTPUT_DIR / "revealed_celltype_and_variance_summary.csv"
    atomic_csv(celltype_summary, celltype_summary_path)
    reveal_manifest = {
        "analysis_version": ANALYSIS_VERSION,
        "model_architecture": MODEL_ARCHITECTURE,
        "model_fingerprint": MODEL_FINGERPRINT,
        "notebook_release_sha256": NOTEBOOK_RELEASE["sha256"],
        "reveal_version": REVEAL_VERSION,
        "run_spec": RUN_SPEC, "profile": PROFILE,
        "selection_status": "revealed_after_all_gates",
        "selection_revealed": True,
        "selection_reveal_utc": datetime.now(timezone.utc).isoformat(),
        "analysis_lock_sha256": ANALYSIS_LOCK_SHA256,
        "posterior_trace_sha256": sha256_file(TRACE_PATH),
        "diagnostics_sha256": sha256_file(diagnostics_path),
        "ppc_adequacy_sha256": sha256_file(ppc_path),
        "feature_inventory_sha256": sha256_file(OUTPUT_DIR / "feature_inventory.csv"),
        "precision_gate_sha256": sha256_file(OUTPUT_DIR / "reveal_precision_gate.json"),
        "results_sha256": sha256_file(results_path),
        "celltype_summary_sha256": sha256_file(celltype_summary_path),
        "rope_grid": list(ROPE_GRID),
        "by_rope": {
            str(rope): {
                "n_selected": value["n_selected"],
                "selected_prefix_mc": value["selected_mc"],
                "first_excluded_prefix_mc": value["next_prefix_mc"],
                "loo_feature_and_cluster_sets_identical": value["loo_pass"],
            }
            for rope, value in rope_assessments.items()
        },
        "v2_provisional_results_status": "seen_but_invalid_due_to_nonconvergence",
    }
    reveal_manifest_path = OUTPUT_DIR / "selection_reveal_manifest.json"
    atomic_json(reveal_manifest, reveal_manifest_path)
    publication_adequacy = {
        "publication_adequacy_pass": True,
        "convergence_pass": True, "ppc_adequacy_pass": True,
        "rope_bfdr_precision_pass": True,
        "selection_reveal_manifest_sha256": sha256_file(reveal_manifest_path),
    }
    atomic_json(publication_adequacy, OUTPUT_DIR / "publication_adequacy.json")
    print(json.dumps(reveal_manifest, indent=2))
else:
    print("Result reveal stage disabled; posterior effects and BFDR were not computed.")


In [ ]:
# 11. Gated global-primary BFDR — no resampling
def global_prefix_mc(families, combined_mask, alpha=MC_ALPHA):
    total_k = int(combined_mask.sum())
    if total_k == 0:
        return {"mean": 0.0, "mcse_guarded": 0.0, "mc_lower": 0.0, "mc_upper": 0.0, "family_parts": []}
    active = []
    for family in families:
        local_mask = combined_mask[family["offset"]:family["offset"] + family["n_features"]]
        k = int(local_mask.sum())
        if k: active.append((family, local_mask, k))
    family_alpha = alpha / len(active)
    parts = []
    for family, local_mask, k in active:
        indices = np.flatnonzero(local_mask)
        series = family["indicator"].isel(feature=indices).mean("feature")
        fallback = float(family["metrics"].iloc[indices]["indicator_ess"].min())
        part = bounded_series_mc(series, fallback, alpha=family_alpha)
        part.update({"family": family["name"], "k": k, "weight": k / total_k})
        parts.append(part)
    return {
        "mean": sum(item["weight"] * item["mean"] for item in parts),
        "mcse_guarded": sum(item["weight"] * item["mcse_guarded"] for item in parts),
        "mc_lower": sum(item["weight"] * item["mc_lower"] for item in parts),
        "mc_upper": sum(item["weight"] * item["mc_upper"] for item in parts),
        "family_parts": parts,
    }

def assess_global_primary_bfdr(families, q=Q_LEVEL):
    probability = np.concatenate([
        family["metrics"]["local_null_probability"].to_numpy() for family in families
    ])
    feature_ids = np.concatenate([family["feature_ids"] for family in families])
    cluster_ids = np.concatenate([family["cluster_ids"] for family in families])
    mask, order, cumulative, k = deterministic_bfdr(probability, feature_ids, q)
    next_index = None if k == len(probability) else int(order[k])
    metrics = pd.concat([family["metrics"] for family in families], ignore_index=True)
    critical = mask | (probability <= 0.10)
    if next_index is not None: critical[next_index] = True
    critical_metrics = metrics.loc[critical]
    indicator_pass = bool(
        (critical_metrics["indicator_ess"] >= ROPE_INDICATOR_ESS_MIN).all()
        and (critical_metrics["indicator_mcse_guarded"] <= ROPE_LOCAL_NULL_MCSE_MAX).all()
    )
    selected_mc = global_prefix_mc(families, mask)
    selected_pass = bool(
        selected_mc["mc_upper"] <= q and selected_mc["mcse_guarded"] <= BFDR_MCSE_MAX
    )
    if next_index is None:
        next_mc, next_pass = None, True
    else:
        next_mask = np.zeros(len(probability), bool); next_mask[order[:k + 1]] = True
        next_mc = global_prefix_mc(families, next_mask)
        next_pass = bool(next_mc["mc_lower"] > q and next_mc["mcse_guarded"] <= BFDR_MCSE_MAX)
    full_features, full_clusters = selection_sets(mask, feature_ids, cluster_ids)
    loo_rows = []
    for family in families:
        for omitted in range(family["indicator"].sizes["chain"]):
            keep = [j for j in range(family["indicator"].sizes["chain"]) if j != omitted]
            perturbed = probability.copy()
            low, high = family["offset"], family["offset"] + family["n_features"]
            perturbed[low:high] = family["indicator"].isel(chain=keep).mean(("chain", "draw")).to_numpy()
            loo_mask, _, _, _ = deterministic_bfdr(perturbed, feature_ids, q)
            feature_set, cluster_set = selection_sets(loo_mask, feature_ids, cluster_ids)
            loo_rows.append({
                "source_family": family["name"], "omitted_chain_position": omitted,
                "feature_set_identical": feature_set == full_features,
                "cluster_set_identical": cluster_set == full_clusters,
            })
    loo = pd.DataFrame(loo_rows)
    loo_pass = bool(loo[["feature_set_identical", "cluster_set_identical"]].to_numpy().all())
    return {
        "pass": bool(indicator_pass and selected_pass and next_pass and loo_pass),
        "mask": mask, "order": order, "cumulative": cumulative, "n_selected": k,
        "indicator_precision_pass": indicator_pass,
        "selected_mc": selected_mc, "selected_bound_pass": selected_pass,
        "next_prefix_mc": next_mc, "next_bound_pass": next_pass,
        "loo": loo, "loo_pass": loo_pass,
        "minimum_critical_indicator_ess": float(critical_metrics["indicator_ess"].min()),
        "maximum_critical_indicator_mcse": float(critical_metrics["indicator_mcse_guarded"].max()),
    }

if RUN_PRIMARY_AGGREGATION:
    if PROFILE != "publication":
        raise ValueError("Aggregation control requires PROFILE='publication'")
    if NOTEBOOK_RELEASE.get("status") != "verified":
        raise RuntimeError(
            "Aggregation requires NOTEBOOK_PATH_OVERRIDE to verify this exact notebook release"
        )
    current_notebook_release = NOTEBOOK_RELEASE["sha256"]
    primary_specs = ("immune_coarse_primary", "nonimmune_primary")

    def revealed_source_directory(spec_name):
        candidates = []
        for candidate_profile in ("publication_long", "publication"):
            candidate = run_directory_for(spec_name, candidate_profile)
            manifest_path = candidate / "selection_reveal_manifest.json"
            if manifest_path.exists():
                manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
                if (
                    manifest.get("selection_status") == "revealed_after_all_gates"
                    and manifest.get("notebook_release_sha256") == current_notebook_release
                ):
                    candidates.append((candidate_profile, candidate))
        if not candidates:
            raise RuntimeError(
                f"{spec_name} has no successful publication or publication_long reveal"
            )
        # A valid publication_long reveal prospectively supersedes its publication
        # predecessor for precision; otherwise use the publication reveal.
        return candidates[0]

    source_records = []
    source_results = []
    source_traces = {}
    for spec_name in primary_specs:
        source_profile, directory = revealed_source_directory(spec_name)
        paths = {
            "trace": directory / "posterior_trace.nc",
            "lock": directory / "analysis_lock.json",
            "diagnostics": directory / "run_diagnostics.json",
            "ppc": directory / "ppc_adequacy.json",
            "precision": directory / "reveal_precision_gate.json",
            "results": directory / "revealed_feature_bmi_results.csv",
            "reveal": directory / "selection_reveal_manifest.json",
        }
        missing = [str(path) for path in paths.values() if not path.exists()]
        if missing:
            raise FileNotFoundError(f"{spec_name} is incomplete: {missing}")
        manifest = json.loads(paths["reveal"].read_text(encoding="utf-8"))
        lock = json.loads(paths["lock"].read_text(encoding="utf-8"))
        diagnostics = json.loads(paths["diagnostics"].read_text(encoding="utf-8"))
        ppc = json.loads(paths["ppc"].read_text(encoding="utf-8"))
        precision = json.loads(paths["precision"].read_text(encoding="utf-8"))
        trace = az.from_netcdf(paths["trace"])
        trace_attrs = {
            key: trace.attrs.get(key, trace.posterior.attrs.get(key))
            for key in (
                "analysis_lock_sha256", "model_fingerprint", "model_architecture",
                "notebook_release_sha256",
            )
        }
        current_hashes = {name: sha256_file(path) for name, path in paths.items()}
        checks = {
            "manifest_analysis_version": manifest.get("analysis_version") == ANALYSIS_VERSION,
            "manifest_architecture": manifest.get("model_architecture") == MODEL_ARCHITECTURE,
            "manifest_notebook_release": manifest.get("notebook_release_sha256") == current_notebook_release,
            "manifest_run_spec": manifest.get("run_spec") == spec_name,
            "manifest_profile": manifest.get("profile") == source_profile,
            "manifest_status": manifest.get("selection_status") == "revealed_after_all_gates",
            "manifest_revealed": manifest.get("selection_revealed") is True,
            "manifest_fingerprint": manifest.get("model_fingerprint") == lock.get("model_fingerprint"),
            "manifest_precision": manifest.get("precision_gate_sha256") == current_hashes["precision"],
            "manifest_trace": manifest.get("posterior_trace_sha256") == current_hashes["trace"],
            "manifest_lock": manifest.get("analysis_lock_sha256") == current_hashes["lock"],
            "manifest_diagnostics": manifest.get("diagnostics_sha256") == current_hashes["diagnostics"],
            "manifest_ppc": manifest.get("ppc_adequacy_sha256") == current_hashes["ppc"],
            "manifest_results": manifest.get("results_sha256") == current_hashes["results"],
            "lock_run_spec": lock.get("run_spec") == spec_name,
            "lock_profile": lock.get("profile") == source_profile,
            "lock_architecture": lock.get("model_architecture") == MODEL_ARCHITECTURE,
            "lock_notebook_release": lock.get("notebook_release_sha256") == current_notebook_release,
            "diagnostic_pass": diagnostics.get("publication_convergence_pass") is True,
            "diagnostic_notebook_verified": diagnostics.get("notebook_release_status") == "verified",
            "diagnostic_notebook_release": diagnostics.get("notebook_release_sha256") == current_notebook_release,
            "diagnostic_profile": diagnostics.get("profile") == source_profile,
            "diagnostic_trace": diagnostics.get("posterior_trace_sha256") == current_hashes["trace"],
            "diagnostic_lock": diagnostics.get("analysis_lock_sha256") == current_hashes["lock"],
            "ppc_pass": ppc.get("ppc_adequacy_pass") is True,
            "ppc_notebook_verified": ppc.get("notebook_release_status") == "verified",
            "ppc_notebook_release": ppc.get("notebook_release_sha256") == current_notebook_release,
            "ppc_profile": ppc.get("profile") == source_profile,
            "ppc_trace": ppc.get("posterior_trace_sha256") == current_hashes["trace"],
            "ppc_lock": ppc.get("analysis_lock_sha256") == current_hashes["lock"],
            "ppc_diagnostics": ppc.get("diagnostics_sha256") == current_hashes["diagnostics"],
            "precision_pass": precision.get("all_displayed_ropes_pass") is True,
            "precision_status": precision.get("selection_status") == "precision_pass",
            "precision_notebook_release": precision.get("notebook_release_sha256") == current_notebook_release,
            "trace_lock": trace_attrs["analysis_lock_sha256"] == current_hashes["lock"],
            "trace_fingerprint": trace_attrs["model_fingerprint"] == lock.get("model_fingerprint"),
            "trace_architecture": trace_attrs["model_architecture"] == MODEL_ARCHITECTURE,
            "trace_notebook_release": trace_attrs["notebook_release_sha256"] == current_notebook_release,
        }
        failed = [name for name, passed in checks.items() if not passed]
        if failed:
            raise RuntimeError(f"{spec_name} reveal provenance failed: {failed}")

        result = pd.read_csv(paths["results"])
        required_result_columns = {"global_feature", "global_cluster_id"}
        if not required_result_columns.issubset(result.columns):
            raise ValueError(f"{spec_name} results lack {required_result_columns - set(result.columns)}")
        if result["global_feature"].duplicated().any():
            raise ValueError(f"{spec_name} has duplicate result feature IDs")
        if result["global_cluster_id"].isna().any():
            raise ValueError(f"{spec_name} has missing cluster IDs")
        posterior_features = np.asarray(
            trace.posterior["feature_bmi_slope"].coords["feature"], str
        )
        if len(np.unique(posterior_features)) != len(posterior_features):
            raise ValueError(f"{spec_name} posterior feature IDs are not unique")
        result_features = result["global_feature"].astype(str).to_numpy()
        if set(result_features) != set(posterior_features):
            raise ValueError(f"{spec_name} result/posterior feature sets differ")
        result = result.set_index("global_feature").loc[posterior_features].reset_index()
        cluster_values = result["global_cluster_id"].astype(str)
        if cluster_values.str.strip().eq("").any() or cluster_values.str.lower().eq("nan").any():
            raise ValueError(f"{spec_name} has empty/invalid cluster IDs")
        source_traces[spec_name] = (trace, result[["global_feature", "global_cluster_id"]].copy())
        result.insert(0, "source_run_spec", spec_name)
        result.insert(1, "primary_family", RUN_SPECS[spec_name]["analysis_family"])
        source_results.append(result)
        source_records.append({
            "run_spec": spec_name,
            "profile": source_profile,
            "directory": str(directory),
            "model_fingerprint": lock["model_fingerprint"],
            "notebook_release_sha256": current_notebook_release,
            "analysis_lock_sha256": current_hashes["lock"],
            "trace_sha256": current_hashes["trace"],
            "diagnostics_sha256": current_hashes["diagnostics"],
            "ppc_adequacy_sha256": current_hashes["ppc"],
            "precision_gate_sha256": current_hashes["precision"],
            "reveal_manifest_sha256": current_hashes["reveal"],
            "results_sha256": current_hashes["results"],
        })

    combined_results = pd.concat(source_results, ignore_index=True)
    assessments = {}
    for rope in ROPE_GRID:
        families = []
        offset = 0
        for spec_name in primary_specs:
            trace, result_clusters = source_traces[spec_name]
            indicator, metrics = rope_indicator_metrics(trace, rope)
            posterior_features = np.asarray(indicator.coords["feature"], str)
            family = {
                "name": spec_name, "indicator": indicator, "metrics": metrics,
                "feature_ids": np.asarray([f"{spec_name}||{x}" for x in posterior_features]),
                "cluster_ids": np.asarray([
                    f"{spec_name}||{x}" for x in result_clusters["global_cluster_id"].astype(str)
                ]),
                "offset": offset, "n_features": len(posterior_features),
            }
            families.append(family)
            offset += family["n_features"]
        assessment = assess_global_primary_bfdr(families)
        assessments[rope] = assessment

    all_pass = bool(all(value["pass"] for value in assessments.values()))
    aggregation_dir = MODEL_ROOT / "global_primary_bfdr_v2p2"
    aggregation_dir.mkdir(parents=True, exist_ok=True)
    gate_record = {
        "analysis_version": ANALYSIS_VERSION,
        "model_architecture": MODEL_ARCHITECTURE,
        "notebook_release_sha256": current_notebook_release,
        "primary_sources": source_records,
        "all_displayed_ropes_pass": all_pass,
        "by_rope": {
            str(rope): {
                "pass": value["pass"],
                "indicator_precision_pass": value["indicator_precision_pass"],
                "selected_bound_pass": value["selected_bound_pass"],
                "next_bound_pass": value["next_bound_pass"],
                "loo_pass": value["loo_pass"],
                "minimum_critical_indicator_ess": value["minimum_critical_indicator_ess"],
                "maximum_critical_indicator_mcse": value["maximum_critical_indicator_mcse"],
            }
            for rope, value in assessments.items()
        },
        "selection_status": "withheld_precision_failed" if not all_pass else "precision_pass",
    }
    gate_path = aggregation_dir / "global_primary_precision_gate.json"
    atomic_json(gate_record, gate_path)
    if not all_pass:
        publication_sources = [
            source["run_spec"] for source in source_records
            if source.get("profile") == "publication"
        ]
        if publication_sources:
            raise RuntimeError(
                "Global-primary BFDR precision failed; no global results/counts "
                "were written. The recorded gate authorizes PROFILE='publication_long' "
                f"for these primary runs: {publication_sources}. Fit/reveal each, "
                "then rerun aggregation."
            )
        raise RuntimeError(
            "Global-primary BFDR precision still failed after all available long "
            "profiles. Stop: no further automatic refit is authorized and no "
            "global results/counts were written."
        )

    for rope, assessment in assessments.items():
        label = str(rope).replace(".", "p")
        combined_results[f"global_primary_bfdr_selected_rope_{label}"] = assessment["mask"]
        for table in (assessment["loo"],):
            atomic_csv(table, aggregation_dir / f"revealed_global_loo_chain_rope_{label}.csv")
    output_path = aggregation_dir / "revealed_global_primary_bfdr.csv"
    atomic_csv(combined_results, output_path)
    summary_rows = []
    for rope, assessment in assessments.items():
        label = str(rope).replace(".", "p")
        selected_column = f"global_primary_bfdr_selected_rope_{label}"
        for family, local in combined_results.groupby("primary_family"):
            selected = strict_bool_series(local[selected_column], selected_column)
            summary_rows.append({
                "rope": rope, "primary_family": family, "n_features": len(local),
                "n_selected_features": int(selected.sum()),
                "n_selected_clusters": int(local.loc[selected, "global_cluster_id"].nunique()),
            })
    summary = pd.DataFrame(summary_rows)
    atomic_csv(summary, aggregation_dir / "revealed_global_primary_bfdr_summary.csv")
    manifest = {
        "analysis_version": ANALYSIS_VERSION,
        "model_architecture": MODEL_ARCHITECTURE,
        "notebook_release_sha256": current_notebook_release,
        "selection_status": "revealed_after_all_gates",
        "selection_revealed": True,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "q_level": Q_LEVEL, "rope_grid": list(ROPE_GRID),
        "primary_sources": source_records,
        "secondary_family_excluded": "immune_fine",
        "reason": "Immune fine is a nested overlapping taxonomy and remains secondary.",
        "precision_gate_sha256": sha256_file(gate_path),
        "results_sha256": sha256_file(output_path),
        "summary_sha256": sha256_file(aggregation_dir / "revealed_global_primary_bfdr_summary.csv"),
        "by_rope": {
            str(rope): {
                "n_selected": value["n_selected"],
                "selected_prefix_mc": value["selected_mc"],
                "first_excluded_prefix_mc": value["next_prefix_mc"],
                "loo_feature_and_cluster_sets_identical": value["loo_pass"],
            }
            for rope, value in assessments.items()
        },
    }
    atomic_json(manifest, aggregation_dir / "global_primary_bfdr_manifest.json")
    display(summary)
    print("Saved:", output_path)
else:
    print("Global primary aggregation stage disabled.")
